# Profilowanie jakościowe - Milestone 1

Cel: sprawdzenie schematu i jakości danych w warstwie bronze za pomocą zapytań SQL w DuckDB.

Dane: pliki Parquet z `data/bronze/` utworzone w notebookach 02. Notebook tylko czyta dane, niczego nie zapisuje.

## 1. Połączenie z DuckDB i widoki

`duckdb.connect()` bez ścieżki tworzy bazę w pamięci, która istnieje tylko na czas działania notebooka.

Na każdy plik Parquet zakładamy widok (`CREATE VIEW`). Widok to zapisane zapytanie, a nie kopia danych: DuckDB czyta plik dopiero przy zapytaniu i tylko te kolumny, których zapytanie potrzebuje. Dzięki temu nie wczytujemy 9 mln wierszy do RAM.

In [1]:
from pathlib import Path
import duckdb

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
BRONZE_DIR = ROOT / 'data' / 'bronze'

TABLES = ['pomiary_2024', 'stacje', 'stanowiska', 'openmeteo_2024']

con = duckdb.connect()

for table in TABLES:
    path = BRONZE_DIR / f'{table}.parquet'
    con.sql(
        f"CREATE OR REPLACE VIEW {table} AS "
        f"SELECT * FROM read_parquet('{path.as_posix()}')"
    )
    print(f'{table:16} {path.stat().st_size / 1_000_000:6.1f} MB')

print(con.sql('SHOW TABLES'))

pomiary_2024       19.2 MB
stacje              0.1 MB
stanowiska          0.1 MB
openmeteo_2024      7.6 MB
┌────────────────┐
│      name      │
│    varchar     │
├────────────────┤
│ openmeteo_2024 │
│ pomiary_2024   │
│ stacje         │
│ stanowiska     │
└────────────────┘



## 2. Schemat tabel

`DESCRIBE SELECT * FROM ...` zwraca nazwy i typy kolumn. DuckDB odczytuje je z metadanych zapisanych w pliku Parquet, bez skanowania danych.

Kolumny `null` i `key` w wyniku pokazują `YES` i `NULL` dla każdej kolumny, bo Parquet nie przechowuje kluczy ani ograniczeń `NOT NULL`. Klucze opisujemy więc pod każdą tabelą, a ich poprawność sprawdzamy w profilowaniu.

### 2.1. `pomiary_2024`

1 wiersz = 1 pomiar jednego stanowiska w jednym okresie (godzina albo doba).

In [2]:
print(con.sql('DESCRIBE SELECT * FROM pomiary_2024'))

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stanowiska   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_stacji       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ czas_usredniania │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ jednostka        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_czas        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ wartosc          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ plik_zrodlowy    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



| Kolumna | Znaczenie |
|---|---|
| `kod_stanowiska` | kod przyrządu pomiarowego: stacja, wskaźnik i czas uśredniania, np. `DsWrocAlWisn-PM10-1g` |
| `kod_stacji` | kod stacji, na której stoi przyrząd |
| `wskaznik` | mierzona substancja, np. `PM10`, `PM2.5`, `NO2`, `O3`, `BaP(PM10)` (benzo(a)piren w pyle PM10) |
| `czas_usredniania` | `1g` - wartość godzinowa, `24g` - wartość dobowa |
| `jednostka` | `ug/m3` dla większości wskaźników, `mg/m3` dla CO, `ng/m3` dla śladowych składników pyłu (np. BaP, As, Cd, Ni) i rtęci |
| `data_czas` | znacznik czasu pomiaru z pliku GIOŚ, bez strefy czasowej; oznacza koniec okresu uśredniania (3.6, 6.2) |
| `wartosc` | zmierzone stężenie; `NULL` oznacza brak pomiaru w tym okresie |
| `plik_zrodlowy` | plik XLSX z warstwy RAW, z którego pochodzi wiersz |

Klucz: `(kod_stanowiska, data_czas)` - jedno stanowisko powinno mieć jeden pomiar w danym okresie. Czy tak jest, sprawdzamy w profilowaniu.

Jednostki różnią się między wskaźnikami, więc statystyki (min, max, średnie) liczymy zawsze osobno dla każdego wskaźnika.

### 2.2. `stacje`

1 wiersz = 1 stacja z metadanych GIOŚ. Arkusz obejmuje wszystkie stacje w historii, także zamknięte przed 2024 r.

In [3]:
print(con.sql('DESCRIBE SELECT * FROM stacje'))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stacji         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_miedzynarodowy │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ nazwa_stacji       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stary_kod_stacji   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_uruchomienia  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ data_zamkniecia    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_stacji         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_obszaru        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rodzaj_stacji      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

| Kolumna | Znaczenie |
|---|---|
| `kod_stacji` | kod stacji GIOŚ, np. `DsWrocAlWisn` (województwo, miejscowość, ulica) |
| `kod_miedzynarodowy` | kod stacji w europejskiej bazie jakości powietrza, np. `PL0810A`; nie każda stacja go ma |
| `nazwa_stacji` | nazwa opisowa |
| `stary_kod_stacji` | poprzedni kod, jeśli stacja go zmieniła |
| `data_uruchomienia`, `data_zamkniecia` | okres działania; pusta data zamknięcia - stacja działa |
| `typ_stacji` | czego wpływ ma mierzyć: `tło`, `komunikacyjna` (ruch drogowy), `przemysłowa` |
| `typ_obszaru` | `miejski`, `podmiejski`, `pozamiejski` |
| `rodzaj_stacji` | rodzaj obiektu, np. `kontenerowa stacjonarna`, `mobilna`, `w budynku` |
| `wojewodztwo`, `miejscowosc`, `adres` | lokalizacja administracyjna |
| `szerokosc`, `dlugosc` | współrzędne WGS84 |

Klucz: `kod_stacji`.

### 2.3. `stanowiska`

1 wiersz = 1 stanowisko, czyli jeden przyrząd mierzący jeden wskaźnik na jednej stacji.

In [4]:
print(con.sql('DESCRIBE SELECT * FROM stanowiska'))

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stanowiska    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_stacji        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik_nazwa    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ czas_usredniania  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_pomiaru       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_uruchomienia │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ data_zamkniecia   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ wojewodztwo       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ strefa    

| Kolumna | Znaczenie |
|---|---|
| `kod_stanowiska` | kod stanowiska, ten sam co w `pomiary_2024` |
| `kod_stacji` | stacja, na której stoi stanowisko |
| `wskaznik`, `wskaznik_nazwa` | kod i pełna nazwa wskaźnika, np. `PM2.5` / `pył zawieszony PM2.5` |
| `czas_usredniania` | zapisany słownie: `1-godzinny`, `24-godzinny`, `2-tygodniowy`, `miesięczny` |
| `typ_pomiaru` | `automatyczny` (analizator, wynik na bieżąco) albo `manualny` (próbka pobrana na filtr i zważona lub zbadana w laboratorium) |
| `data_uruchomienia`, `data_zamkniecia` | okres działania stanowiska |
| `wojewodztwo` | województwo |
| `strefa` | strefa oceny jakości powietrza, np. aglomeracja albo pozostały obszar województwa |

Klucz: `kod_stanowiska`.

Czas uśredniania jest zapisany inaczej niż w `pomiary_2024` (`1-godzinny` zamiast `1g`). Dlatego pomiary łączymy ze stanowiskami po `kod_stanowiska`, a nie po parze wskaźnik i czas uśredniania.

### 2.4. `openmeteo_2024`

1 wiersz = pogoda dla 1 stacji mierzącej PM2.5 w 1 godzinie.

In [5]:
print(con.sql('DESCRIBE SELECT * FROM openmeteo_2024'))

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stacji           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_czas_utc        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ temperature_2m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ relative_humidity_2m │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ precipitation        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ surface_pressure     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_speed_10m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_direction_10m   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ siatka_szerokosc     │ DOUBLE      │ YES     │ NUL

| Kolumna | Znaczenie |
|---|---|
| `kod_stacji` | kod stacji GIOŚ, wzięty z nazwy pliku JSON |
| `data_czas_utc` | godzina w UTC |
| `temperature_2m` | temperatura powietrza 2 m nad ziemią, °C |
| `relative_humidity_2m` | wilgotność względna 2 m nad ziemią, % |
| `precipitation` | suma opadu z poprzedniej godziny, mm |
| `surface_pressure` | ciśnienie na poziomie terenu (nie przeliczone do poziomu morza), hPa |
| `wind_speed_10m` | prędkość wiatru 10 m nad ziemią, km/h |
| `wind_direction_10m` | kierunek, z którego wieje wiatr, ° (0 = północ) |
| `siatka_szerokosc`, `siatka_dlugosc` | punkt siatki modelu pogodowego najbliższy stacji; to nie są dokładne współrzędne stacji |
| `wysokosc_m` | wysokość terenu n.p.m. przyjęta przez Open-Meteo dla współrzędnych stacji, a nie dla komórki siatki (5.4) |
| `plik_zrodlowy` | plik JSON z warstwy RAW |

Klucz: `(kod_stacji, data_czas_utc)`.

To dane z modelu meteorologicznego (reanaliza), a nie pomiar na stacji GIOŚ.

### 2.5. Relacje między tabelami

```text
pomiary_2024   ──(kod_stanowiska)──►  stanowiska  ──(kod_stacji)──►  stacje
openmeteo_2024 ─────────────────────(kod_stacji)──────────────────►  stacje
pomiary_2024   ◄──(kod_stacji + czas)──►  openmeteo_2024     [Milestone 2]
```

- `pomiary_2024.kod_stanowiska` → `stanowiska.kod_stanowiska`
- `pomiary_2024.kod_stacji`, `stanowiska.kod_stacji`, `openmeteo_2024.kod_stacji` → `stacje.kod_stacji`
- pomiary z pogodą łączą się po stacji i czasie. GIOŚ zapisuje czas bez strefy, a Open-Meteo w UTC, więc zgodność czasu sprawdzamy w profilowaniu przed złączeniem w warstwie silver (Milestone 2).

## 3. Profil pomiarów (`pomiary_2024`)

Wszystkie statystyki liczone zapytaniami SQL bezpośrednio na pliku Parquet.

### 3.1. Liczba rekordów

In [6]:
con.sql('''
SELECT
    count(*) AS rekordy,
    count(DISTINCT kod_stacji) AS stacje,
    count(DISTINCT kod_stanowiska) AS stanowiska,
    count(DISTINCT wskaznik) AS wskazniki,
    count(DISTINCT (wskaznik, czas_usredniania)) AS wskazniki_z_czasem,
    count(DISTINCT plik_zrodlowy) AS pliki
FROM pomiary_2024
''').show(max_rows=40, max_width=250)

┌─────────┬────────┬────────────┬───────────┬────────────────────┬───────┐
│ rekordy │ stacje │ stanowiska │ wskazniki │ wskazniki_z_czasem │ pliki │
│  int64  │ int64  │   int64    │   int64   │       int64        │ int64 │
├─────────┼────────┼────────────┼───────────┼────────────────────┼───────┤
│ 8952726 │    286 │       1759 │        60 │                 64 │    28 │
└─────────┴────────┴────────────┴───────────┴────────────────────┴───────┘



Rozbicie na wskaźnik i czas uśredniania. `rekordy` to wszystkie wiersze, `pomiary` - wiersze z wartością, `braki` - wiersze z `NULL` w `wartosc`, czyli okresy bez pomiaru.

In [7]:
con.sql('''
SELECT
    wskaznik,
    czas_usredniania,
    jednostka,
    count(DISTINCT kod_stanowiska) AS stanowiska,
    count(*) AS rekordy,
    count(wartosc) AS pomiary,
    count(*) - count(wartosc) AS braki,
    round(100 * count(wartosc) / count(*), 1) AS kompletnosc_proc
FROM pomiary_2024
GROUP BY ALL
ORDER BY rekordy DESC
''').show(max_rows=70, max_width=250)

┌────────────────────┬──────────────────┬───────────┬────────────┬─────────┬─────────┬───────┬──────────────────┐
│      wskaznik      │ czas_usredniania │ jednostka │ stanowiska │ rekordy │ pomiary │ braki │ kompletnosc_proc │
│      varchar       │     varchar      │  varchar  │   int64    │  int64  │  int64  │ int64 │      double      │
├────────────────────┼──────────────────┼───────────┼────────────┼─────────┼─────────┼───────┼──────────────────┤
│ PM10               │ 1g               │ ug/m3     │        179 │ 1572336 │ 1519684 │ 52652 │             96.7 │
│ NO                 │ 1g               │ ug/m3     │        134 │ 1177056 │ 1135235 │ 41821 │             96.4 │
│ NO2                │ 1g               │ ug/m3     │        134 │ 1177056 │ 1135319 │ 41737 │             96.5 │
│ NOx                │ 1g               │ ug/m3     │        115 │ 1010160 │  973833 │ 36327 │             96.4 │
│ O3                 │ 1g               │ ug/m3     │         98 │  860832 │  839875 │ 2

Wskaźników jest 60, a par wskaźnik + czas uśredniania 64, bo NO2, SO2, PM10 i PM2.5 są mierzone zarówno godzinowo, jak i dobowo. Plik `2024_PrekursoryZielonka_1g.xlsx` zawiera 29 różnych lotnych związków organicznych z jednej stacji (Bory Tucholskie), a `2024_Jony_PM25_24g.xlsx` - 10 składników pyłu PM2.5.

### 3.2. Braki danych

Liczba `NULL` w każdej kolumnie. Braki w kluczach albo w czasie oznaczałyby błąd konwersji do bronze.

In [8]:
con.sql('''
SELECT
    count(*) - count(kod_stanowiska) AS kod_stanowiska,
    count(*) - count(kod_stacji) AS kod_stacji,
    count(*) - count(wskaznik) AS wskaznik,
    count(*) - count(czas_usredniania) AS czas_usredniania,
    count(*) - count(jednostka) AS jednostka,
    count(*) - count(data_czas) AS data_czas,
    count(*) - count(wartosc) AS wartosc,
    count(*) - count(plik_zrodlowy) AS plik_zrodlowy
FROM pomiary_2024
''').show(max_rows=40, max_width=250)

┌────────────────┬────────────┬──────────┬──────────────────┬───────────┬───────────┬─────────┬───────────────┐
│ kod_stanowiska │ kod_stacji │ wskaznik │ czas_usredniania │ jednostka │ data_czas │ wartosc │ plik_zrodlowy │
│     int64      │   int64    │  int64   │      int64       │   int64   │   int64   │  int64  │     int64     │
├────────────────┼────────────┼──────────┼──────────────────┼───────────┼───────────┼─────────┼───────────────┤
│              0 │          0 │        0 │                0 │         0 │         0 │  470793 │             0 │
└────────────────┴────────────┴──────────┴──────────────────┴───────────┴───────────┴─────────┴───────────────┘



Braki występują tylko w `wartosc`. Kompletność dla wskaźników jest w tabeli w 3.1.

Dla PM2.5 (zmienna docelowa modelu) sprawdzamy kompletność dokładniej: na stanowisko i na miesiąc. Próg 90% to minimalna kompletność wymagana dla pomiarów stałych w dyrektywie UE 2008/50/WE.

In [9]:
con.sql('''
WITH kompletnosc AS (
    SELECT
        kod_stanowiska,
        100 * count(wartosc) / count(*) AS proc
    FROM pomiary_2024
    WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
    GROUP BY kod_stanowiska
)
SELECT
    count(*) AS stanowiska,
    round(min(proc), 1) AS min_proc,
    round(median(proc), 1) AS mediana_proc,
    count(*) FILTER (WHERE proc < 90) AS ponizej_90_proc,
    count(*) FILTER (WHERE proc < 75) AS ponizej_75_proc
FROM kompletnosc
''').show(max_rows=40, max_width=250)

┌────────────┬──────────┬──────────────┬─────────────────┬─────────────────┐
│ stanowiska │ min_proc │ mediana_proc │ ponizej_90_proc │ ponizej_75_proc │
│   int64    │  double  │    double    │      int64      │      int64      │
├────────────┼──────────┼──────────────┼─────────────────┼─────────────────┤
│         96 │     42.7 │         98.6 │               3 │               1 │
└────────────┴──────────┴──────────────┴─────────────────┴─────────────────┘



In [10]:
con.sql('''
SELECT
    kod_stanowiska,
    count(*) - count(wartosc) AS braki,
    round(100 * count(wartosc) / count(*), 1) AS kompletnosc_proc
FROM pomiary_2024
WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
GROUP BY kod_stanowiska
ORDER BY kompletnosc_proc
LIMIT 10
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬───────┬──────────────────┐
│      kod_stanowiska      │ braki │ kompletnosc_proc │
│         varchar          │ int64 │      double      │
├──────────────────────────┼───────┼──────────────────┤
│ MzSiedKonars-PM2.5-1g    │  5034 │             42.7 │
│ PdBialUpalna-PM2.5-1g    │   953 │             89.2 │
│ LdRadomsRoln-PM2.5-1g    │   901 │             89.7 │
│ SkPolaRuszcz-PM2.5-1g    │   874 │             90.1 │
│ KpBydPlPozna-PM2.5-1g    │   830 │             90.6 │
│ PkJasloSikor-PM2.5-1g    │   744 │             91.5 │
│ KpWloclOkrze-PM2.5-1g    │   686 │             92.2 │
│ PdGrajewoWPoMOB-PM2.5-1g │   678 │             92.3 │
│ PkMielBierna-PM2.5-1g    │   646 │             92.6 │
│ WmDzialPolnaMOB-PM2.5-1g │   588 │             93.3 │
└──────────────────────────┴───────┴──────────────────┘
  10 rows                                   3 columns



Kompletność PM2.5 na miesiąc. Od czasu odejmujemy godzinę, bo GIOŚ oznacza pomiar godziną końca okresu (pomiar z `2025-01-01 00:00` dotyczy ostatniej godziny grudnia - patrz 3.6).

In [11]:
con.sql('''
SELECT
    month(data_czas - INTERVAL 1 HOUR) AS miesiac,
    count(*) - count(wartosc) AS braki,
    round(100 * count(wartosc) / count(*), 1) AS kompletnosc_proc
FROM pomiary_2024
WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
GROUP BY miesiac
ORDER BY miesiac
''').show(max_rows=40, max_width=250)

┌─────────┬───────┬──────────────────┐
│ miesiac │ braki │ kompletnosc_proc │
│  int64  │ int64 │      double      │
├─────────┼───────┼──────────────────┤
│       1 │  2514 │             96.5 │
│       2 │   761 │             98.9 │
│       3 │   702 │             99.0 │
│       4 │   927 │             98.7 │
│       5 │  2486 │             96.5 │
│       6 │  2131 │             96.9 │
│       7 │  2260 │             96.8 │
│       8 │  2651 │             96.3 │
│       9 │  2723 │             96.1 │
│      10 │  1892 │             97.4 │
│      11 │  1795 │             97.4 │
│      12 │  4774 │             93.3 │
└─────────┴───────┴──────────────────┘
  12 rows                  3 columns



### 3.3. Statystyki wartości na wskaźnik

Min, max, średnia i percentyle liczone osobno dla każdego wskaźnika, bo jednostki i typowe poziomy są różne. `max_do_p99_9` to stosunek maksimum do percentyla 99,9: duża wartość oznacza pojedyncze wartości daleko od reszty rozkładu.

In [12]:
con.sql('''
SELECT
    wskaznik,
    czas_usredniania,
    jednostka,
    min(wartosc) AS min,
    round(quantile_cont(wartosc, 0.01), 2) AS p1,
    round(median(wartosc), 2) AS p50,
    round(avg(wartosc), 2) AS srednia,
    round(quantile_cont(wartosc, 0.99), 2) AS p99,
    round(quantile_cont(wartosc, 0.999), 2) AS p99_9,
    max(wartosc) AS max,
    round(max(wartosc) / nullif(quantile_cont(wartosc, 0.999), 0), 1) AS max_do_p99_9
FROM pomiary_2024
GROUP BY ALL
ORDER BY wskaznik, czas_usredniania
''').show(max_rows=70, max_width=250)

┌────────────────────┬──────────────────┬───────────┬────────┬────────┬────────┬─────────┬────────┬────────┬─────────┬──────────────┐
│      wskaznik      │ czas_usredniania │ jednostka │  min   │   p1   │  p50   │ srednia │  p99   │ p99_9  │   max   │ max_do_p99_9 │
│      varchar       │     varchar      │  varchar  │ double │ double │ double │ double  │ double │ double │ double  │    double    │
├────────────────────┼──────────────────┼───────────┼────────┼────────┼────────┼─────────┼────────┼────────┼─────────┼──────────────┤
│ 123trimetylobenzen │ 1g               │ ug/m3     │    0.0 │   0.01 │   0.14 │    0.19 │   0.83 │   1.14 │    1.25 │          1.1 │
│ 124trimetylobenzen │ 1g               │ ug/m3     │    0.0 │   0.02 │   1.32 │    1.24 │   2.73 │   2.89 │    2.91 │          1.0 │
│ 135trimetylobenzen │ 1g               │ ug/m3     │    0.0 │   0.04 │    0.4 │    0.52 │   2.24 │   3.75 │    3.96 │          1.1 │
│ 13butadien         │ 1g               │ ug/m3     │    0.0 │

### 3.4. Podejrzane wartości

**Wartości ujemne i zera.** Stężenie nie może być ujemne. Zero dokładnie `0.0` jest możliwe dla niektórych gazów przy bardzo czystym powietrzu, ale dla pyłu oznacza raczej wartość poniżej progu wykrywalności przyrządu albo jego awarię.

In [13]:
con.sql('''
SELECT
    wskaznik,
    czas_usredniania,
    count(wartosc) AS pomiary,
    count(*) FILTER (WHERE wartosc < 0) AS ujemne,
    count(*) FILTER (WHERE wartosc = 0) AS zera,
    round(100 * count(*) FILTER (WHERE wartosc = 0) / count(wartosc), 2) AS zera_proc
FROM pomiary_2024
GROUP BY ALL
HAVING ujemne > 0 OR zera > 0
ORDER BY zera DESC
''').show(max_rows=70, max_width=250)

┌────────────────────┬──────────────────┬─────────┬────────┬───────┬───────────┐
│      wskaznik      │ czas_usredniania │ pomiary │ ujemne │ zera  │ zera_proc │
│      varchar       │     varchar      │  int64  │ int64  │ int64 │  double   │
├────────────────────┼──────────────────┼─────────┼────────┼───────┼───────────┤
│ NO                 │ 1g               │ 1135235 │      0 │ 14856 │      1.31 │
│ SO2                │ 1g               │  665147 │      0 │  6489 │      0.98 │
│ C6H6               │ 1g               │  535548 │      0 │  2382 │      0.44 │
│ PM10               │ 1g               │ 1519684 │      0 │   990 │      0.07 │
│ PM2.5              │ 1g               │  817648 │      0 │   728 │      0.09 │
│ O3                 │ 1g               │  839875 │      0 │   107 │      0.01 │
│ Ni(PM10)           │ 24g              │   20431 │      0 │   105 │      0.51 │
│ NO2                │ 1g               │ 1135319 │      0 │    64 │      0.01 │
│ oksylen            │ 1g   

In [14]:
con.sql('''
SELECT kod_stanowiska, data_czas, wartosc, jednostka
FROM pomiary_2024
WHERE wartosc < 0
''').show(max_rows=40, max_width=250)

┌────────────────────────────┬─────────────────────┬─────────┬───────────┐
│       kod_stanowiska       │      data_czas      │ wartosc │ jednostka │
│          varchar           │      timestamp      │ double  │  varchar  │
├────────────────────────────┼─────────────────────┼─────────┼───────────┤
│ WmPuszczaBor-OC(PM2.5)-24g │ 2024-12-16 00:00:00 │   -0.35 │ ug/m3     │
│ WmPuszczaBor-OC(PM2.5)-24g │ 2024-12-17 00:00:00 │   -0.37 │ ug/m3     │
└────────────────────────────┴─────────────────────┴─────────┴───────────┘



Zera w PM2.5 i PM10 nie są rozłożone równo - skupiają się na kilku stanowiskach:

In [15]:
con.sql('''
SELECT
    kod_stanowiska,
    count(*) FILTER (WHERE wartosc = 0) AS zera,
    count(wartosc) AS pomiary
FROM pomiary_2024
WHERE wskaznik IN ('PM2.5', 'PM10') AND czas_usredniania = '1g'
GROUP BY kod_stanowiska
HAVING zera > 0
ORDER BY zera DESC
LIMIT 8
''').show(max_rows=40, max_width=250)

┌───────────────────────┬───────┬─────────┐
│    kod_stanowiska     │ zera  │ pomiary │
│        varchar        │ int64 │  int64  │
├───────────────────────┼───────┼─────────┤
│ PdBorsukowiz-PM2.5-1g │   427 │    8681 │
│ PdBorsukowiz-PM10-1g  │   306 │    8681 │
│ PdSuwPulask2-PM2.5-1g │   198 │    8473 │
│ DsDziePilsud-PM10-1g  │   185 │    8605 │
│ PdBialWarsza-PM10-1g  │   180 │    8627 │
│ DsTrzebniMaj-PM10-1g  │   139 │    8690 │
│ PdBialUpalna-PM2.5-1g │    91 │    7831 │
│ DsLegAlRzecz-PM10-1g  │    71 │    7780 │
└───────────────────────┴───────┴─────────┘



**Spójność fizyczna między wskaźnikami.** Zamiast arbitralnych progów sprawdzamy zależności, które muszą zachodzić z definicji:

- PM2.5 to część PM10 (pył o średnicy do 2,5 µm zawiera się w pyle do 10 µm), więc na tej samej stacji w tej samej godzinie PM2.5 ≤ PM10;
- NOx to suma NO i NO2 (przeliczona na NO2), więc NO2 ≤ NOx.

Każda stacja ma najwyżej jedno godzinowe stanowisko danego wskaźnika, więc porównanie w obrębie stacji i godziny jest jednoznaczne.

In [16]:
con.sql('''
WITH pyl AS (
    SELECT
        kod_stacji,
        data_czas,
        max(wartosc) FILTER (WHERE wskaznik = 'PM2.5') AS pm25,
        max(wartosc) FILTER (WHERE wskaznik = 'PM10') AS pm10
    FROM pomiary_2024
    WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
    GROUP BY kod_stacji, data_czas
)
SELECT
    count(*) FILTER (WHERE pm25 IS NOT NULL AND pm10 IS NOT NULL) AS godziny_z_oboma,
    count(*) FILTER (WHERE pm25 > pm10) AS pm25_wieksze_od_pm10,
    count(DISTINCT kod_stacji) FILTER (WHERE pm25 > pm10) AS stacje,
    round(max(pm25 - pm10), 1) AS max_nadwyzka
FROM pyl
''').show(max_rows=40, max_width=250)

┌─────────────────┬──────────────────────┬────────┬──────────────┐
│ godziny_z_oboma │ pm25_wieksze_od_pm10 │ stacje │ max_nadwyzka │
│      int64      │        int64         │ int64  │    double    │
├─────────────────┼──────────────────────┼────────┼──────────────┤
│          773061 │                 1003 │      8 │          6.4 │
└─────────────────┴──────────────────────┴────────┴──────────────┘



In [17]:
con.sql('''
WITH tlenki AS (
    SELECT
        kod_stacji,
        data_czas,
        max(wartosc) FILTER (WHERE wskaznik = 'NO2') AS no2,
        max(wartosc) FILTER (WHERE wskaznik = 'NOx') AS nox
    FROM pomiary_2024
    WHERE czas_usredniania = '1g' AND wskaznik IN ('NO2', 'NOx')
    GROUP BY kod_stacji, data_czas
)
SELECT
    count(*) FILTER (WHERE no2 IS NOT NULL AND nox IS NOT NULL) AS godziny_z_oboma,
    count(*) FILTER (WHERE no2 > nox) AS no2_wieksze_od_nox,
    count(*) FILTER (WHERE no2 > nox + 5) AS wieksze_o_ponad_5,
    count(DISTINCT kod_stacji) FILTER (WHERE no2 > nox) AS stacje,
    round(max(no2 - nox), 1) AS max_nadwyzka
FROM tlenki
''').show(max_rows=40, max_width=250)

┌─────────────────┬────────────────────┬───────────────────┬────────┬──────────────┐
│ godziny_z_oboma │ no2_wieksze_od_nox │ wieksze_o_ponad_5 │ stacje │ max_nadwyzka │
│      int64      │       int64        │       int64       │ int64  │    double    │
├─────────────────┼────────────────────┼───────────────────┼────────┼──────────────┤
│          973788 │               5220 │                10 │     34 │          9.9 │
└─────────────────┴────────────────────┴───────────────────┴────────┴──────────────┘



PM2.5 przekracza PM10 w 1003 godzinach na 8 stacjach, ale najwyżej o 6,4 µg/m³. NO2 przekracza NOx w 5220 godzinach na 34 stacjach, z czego tylko 10 razy o więcej niż 5 µg/m³ (maksymalnie 9,9). Oba wskaźniki mierzą osobne przyrządy, więc to drobne niezgodności na poziomie kilku µg/m³, a nie wartości wielokrotnie przekraczające fizyczne ograniczenie.

**Wartości skrajne.** Liczba wartości powyżej p99,9 wynosi z definicji około 0,1% pomiarów, więc sama w sobie nic nie mówi. Zamiast niej pokazujemy największe wartości z datą - pozwala to odróżnić pojedynczy odczyt od epizodu trwającego kilka godzin albo obejmującego kilka stacji naraz.

In [18]:
con.sql('''
SELECT wskaznik, kod_stanowiska, data_czas, wartosc, jednostka
FROM pomiary_2024
WHERE czas_usredniania = '1g'
    AND wskaznik IN ('PM2.5', 'PM10', 'SO2', 'NOx')
    AND wartosc IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY wskaznik ORDER BY wartosc DESC) <= 5
ORDER BY wskaznik, wartosc DESC
''').show(max_rows=40, max_width=250)

┌──────────┬───────────────────────┬─────────────────────┬─────────┬───────────┐
│ wskaznik │    kod_stanowiska     │      data_czas      │ wartosc │ jednostka │
│ varchar  │        varchar        │      timestamp      │ double  │  varchar  │
├──────────┼───────────────────────┼─────────────────────┼─────────┼───────────┤
│ NOx      │ SlCzestoArmK-NOx-1g   │ 2024-11-06 18:00:00 │  1104.5 │ ug/m3     │
│ NOx      │ DsWrocAlWisn-NOx-1g   │ 2024-12-02 08:00:00 │  1036.4 │ ug/m3     │
│ NOx      │ SlCzestoArmK-NOx-1g   │ 2024-11-06 19:00:00 │  1031.9 │ ug/m3     │
│ NOx      │ KpWloclOkrze-NOx-1g   │ 2024-11-05 18:00:00 │   984.1 │ ug/m3     │
│ NOx      │ DsWrocAlWisn-NOx-1g   │ 2024-12-02 09:00:00 │   964.0 │ ug/m3     │
│ PM10     │ WpMosinaCzer-PM10-1g  │ 2024-01-10 21:00:00 │   526.5 │ ug/m3     │
│ PM10     │ WpMosinaCzer-PM10-1g  │ 2024-01-10 22:00:00 │   513.4 │ ug/m3     │
│ PM10     │ SlRybniBorki-PM10-1g  │ 2024-01-10 22:00:00 │   472.2 │ ug/m3     │
│ PM10     │ PdAugustowUz-PM

Najwyższe wartości nie są pojedynczymi odczytami: PM2.5 i PM10 osiągają maksima w kilku stacjach jednocześnie w dniach 8-10 stycznia, a skrajne SO2 (1353,8 µg/m³, 28 razy więcej niż p99,9) występuje w tych samych godzinach 27 lutego na dwóch stacjach w Szczecinie. To wzorzec epizodu obejmującego obszar, a nie awarii jednego przyrządu.

### 3.5. Duplikaty

Klucz `(kod_stanowiska, data_czas)` powinien być unikalny.

In [19]:
con.sql('''
SELECT
    count(*) AS rekordy,
    count(DISTINCT (kod_stanowiska, data_czas)) AS unikalne_klucze,
    count(*) - count(DISTINCT (kod_stanowiska, data_czas)) AS nadmiarowe_wiersze
FROM pomiary_2024
''').show(max_rows=40, max_width=250)

┌─────────┬─────────────────┬────────────────────┐
│ rekordy │ unikalne_klucze │ nadmiarowe_wiersze │
│  int64  │      int64      │       int64        │
├─────────┼─────────────────┼────────────────────┤
│ 8952726 │         8943942 │               8784 │
└─────────┴─────────────────┴────────────────────┘



In [20]:
con.sql('''
WITH dubel AS (
    SELECT
        kod_stanowiska,
        data_czas,
        list(plik_zrodlowy ORDER BY plik_zrodlowy) AS pliki,
        min(wartosc) IS DISTINCT FROM max(wartosc)
            OR count(wartosc) NOT IN (0, count(*)) AS rozne_wartosci
    FROM pomiary_2024
    GROUP BY kod_stanowiska, data_czas
    HAVING count(*) > 1
)
SELECT
    kod_stanowiska,
    pliki,
    count(*) AS zdublowane_okresy,
    count(*) FILTER (WHERE rozne_wartosci) AS z_roznymi_wartosciami
FROM dubel
GROUP BY ALL
''').show(max_rows=40, max_width=250)

┌──────────────────────┬──────────────────────────────────────────────────────┬───────────────────┬───────────────────────┐
│    kod_stanowiska    │                        pliki                         │ zdublowane_okresy │ z_roznymi_wartosciami │
│       varchar        │                      varchar[]                       │       int64       │         int64         │
├──────────────────────┼──────────────────────────────────────────────────────┼───────────────────┼───────────────────────┤
│ KpZielBoryTu-C6H6-1g │ [2024_C6H6_1g.xlsx, 2024_PrekursoryZielonka_1g.xlsx] │              8784 │                     0 │
└──────────────────────┴──────────────────────────────────────────────────────┴───────────────────┴───────────────────────┘



Wszystkie duplikaty to jedno stanowisko, które występuje w dwóch plikach z identycznymi wartościami.

Do analiz szeregów czasowych (3.6-3.8) potrzebujemy jednego wiersza na stanowisko i godzinę, inaczej `lag`/`lead` porównywałyby wiersz z jego kopią. Zakładamy więc widok bez duplikatów. To tylko widok na potrzeby profilowania - dane w bronze zostają bez zmian.

In [21]:
con.sql('''
CREATE OR REPLACE VIEW pomiary_bez_dubli AS
SELECT *
FROM pomiary_2024
QUALIFY row_number() OVER (PARTITION BY kod_stanowiska, data_czas ORDER BY plik_zrodlowy) = 1
''')

con.sql('SELECT count(*) AS rekordy FROM pomiary_bez_dubli').show()

┌─────────┐
│ rekordy │
│  int64  │
├─────────┤
│ 8943942 │
└─────────┘



### 3.6. Czas

Zakres dat, wpisy spoza 2024 r., wpisy dobowe niebędące północą i znaczniki niebędące pełną godziną.

In [22]:
con.sql('''
SELECT
    czas_usredniania,
    min(data_czas) AS pierwszy,
    max(data_czas) AS ostatni,
    count(*) FILTER (WHERE year(data_czas) <> 2024) AS poza_2024,
    count(*) FILTER (WHERE data_czas = TIMESTAMP '2024-01-01 00:00:00') AS o_polnocy_1_stycznia,
    count(*) FILTER (WHERE czas_usredniania = '24g' AND data_czas::TIME <> TIME '00:00:00') AS dobowe_nie_o_polnocy,
    count(*) FILTER (WHERE minute(data_czas) <> 0 OR second(data_czas) <> 0) AS nie_pelna_godzina
FROM pomiary_2024
GROUP BY czas_usredniania
''').show(max_rows=40, max_width=250)

┌──────────────────┬─────────────────────┬─────────────────────┬───────────┬──────────────────────┬──────────────────────┬───────────────────┐
│ czas_usredniania │      pierwszy       │       ostatni       │ poza_2024 │ o_polnocy_1_stycznia │ dobowe_nie_o_polnocy │ nie_pelna_godzina │
│     varchar      │      timestamp      │      timestamp      │   int64   │        int64         │        int64         │       int64       │
├──────────────────┼─────────────────────┼─────────────────────┼───────────┼──────────────────────┼──────────────────────┼───────────────────┤
│ 24g              │ 2024-01-01 00:00:00 │ 2024-12-31 00:00:00 │         0 │                  773 │                    0 │                 0 │
│ 1g               │ 2024-01-01 01:00:00 │ 2025-01-01 00:00:00 │       987 │                    0 │                    0 │                 0 │
└──────────────────┴─────────────────────┴─────────────────────┴───────────┴──────────────────────┴──────────────────────┴───────────────────┘

Pomiary godzinowe zaczynają się o `2024-01-01 01:00`, a kończą o `2025-01-01 00:00`. Wpisy z 2025 r. (987 = jeden na każde z 986 stanowisk godzinowych plus kopia zdublowanego stanowiska z 3.5) nie są więc błędem: GIOŚ oznacza pomiar godziną końca okresu uśredniania, czyli `01:00` to średnia z 00:00-01:00. Trzeba to uwzględnić przy łączeniu z pogodą, w której znacznik oznacza początek godziny.

Czy każde stanowisko ma pełną siatkę czasu (8784 godziny w roku przestępnym albo 366 dób)?

In [23]:
con.sql('''
WITH na_stanowisko AS (
    SELECT kod_stanowiska, czas_usredniania, count(*) AS rekordy
    FROM pomiary_bez_dubli
    GROUP BY ALL
)
SELECT czas_usredniania, rekordy AS rekordy_na_stanowisko, count(*) AS stanowiska
FROM na_stanowisko
GROUP BY ALL
ORDER BY ALL
''').show(max_rows=40, max_width=250)

┌──────────────────┬───────────────────────┬────────────┐
│ czas_usredniania │ rekordy_na_stanowisko │ stanowiska │
│     varchar      │         int64         │   int64    │
├──────────────────┼───────────────────────┼────────────┤
│ 1g               │                  8784 │        986 │
│ 24g              │                   366 │        773 │
└──────────────────┴───────────────────────┴────────────┘



Każde stanowisko ma wiersz dla każdego okresu, bo plik GIOŚ to pełna tabela godzin × stanowisk. Brakujący pomiar to więc nie brakujący wiersz, tylko `NULL` w `wartosc`.

**Luki w szeregu.** Żeby zmierzyć, jak długie są przerwy w pomiarach, dzielimy każdy szereg godzinowy na ciągi kolejnych godzin o tej samej wartości (metoda *gaps and islands*): nowy ciąg zaczyna się, gdy wartość różni się od poprzedniej godziny. Ciąg `NULL` to luka, a ciąg tej samej liczby - stała wartość (wykorzystamy go w 3.7).

Wynik zapisujemy jako tabelę tymczasową w pamięci, bo korzystają z niego dwie kolejne analizy.

In [24]:
con.sql('''
CREATE OR REPLACE TEMP TABLE ciagi AS
WITH kolejne AS (
    SELECT
        kod_stanowiska,
        wskaznik,
        data_czas,
        wartosc,
        CASE WHEN wartosc IS NOT DISTINCT FROM lag(wartosc) OVER w THEN 0 ELSE 1 END AS nowy_ciag
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g'
    WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
),
numerowane AS (
    SELECT *, sum(nowy_ciag) OVER (PARTITION BY kod_stanowiska ORDER BY data_czas) AS nr_ciagu
    FROM kolejne
)
SELECT
    kod_stanowiska,
    wskaznik,
    nr_ciagu,
    any_value(wartosc) AS wartosc,
    min(data_czas) AS poczatek,
    max(data_czas) AS koniec,
    count(*) AS godziny
FROM numerowane
GROUP BY kod_stanowiska, wskaznik, nr_ciagu
''')

con.sql('SELECT count(*) AS ciagi FROM ciagi').show()

┌─────────┐
│  ciagi  │
│  int64  │
├─────────┤
│ 7774651 │
└─────────┘



Luki w głównych wskaźnikach godzinowych:

In [25]:
con.sql('''
SELECT
    wskaznik,
    count(*) AS luki,
    count(*) FILTER (WHERE godziny >= 24) AS od_doby,
    count(*) FILTER (WHERE godziny >= 24 * 7) AS od_tygodnia,
    max(godziny) AS najdluzsza_h
FROM ciagi
WHERE wartosc IS NULL AND wskaznik IN ('PM10', 'PM2.5', 'NO2', 'NO', 'NOx', 'O3', 'SO2', 'CO', 'C6H6')
GROUP BY wskaznik
ORDER BY wskaznik
''').show(max_rows=40, max_width=250)

┌──────────┬───────┬─────────┬─────────────┬──────────────┐
│ wskaznik │ luki  │ od_doby │ od_tygodnia │ najdluzsza_h │
│ varchar  │ int64 │  int64  │    int64    │    int64     │
├──────────┼───────┼─────────┼─────────────┼──────────────┤
│ C6H6     │  4374 │     127 │          22 │         2905 │
│ CO       │  2051 │      57 │           3 │         2896 │
│ NO       │  5943 │     151 │          23 │         6274 │
│ NO2      │  5893 │     151 │          23 │         6274 │
│ NOx      │  4803 │     114 │          19 │         6274 │
│ O3       │  2368 │     131 │          15 │         2896 │
│ PM10     │  4051 │     288 │          45 │         4993 │
│ PM2.5    │  2721 │     127 │          29 │         4993 │
│ SO2      │  3714 │      84 │          14 │         2905 │
└──────────┴───────┴─────────┴─────────────┴──────────────┘



In [26]:
con.sql('''
SELECT kod_stanowiska, poczatek, koniec, godziny
FROM ciagi
WHERE wartosc IS NULL AND wskaznik = 'PM2.5'
ORDER BY godziny DESC
LIMIT 8
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬─────────────────────┬─────────────────────┬─────────┐
│      kod_stanowiska      │      poczatek       │       koniec        │ godziny │
│         varchar          │      timestamp      │      timestamp      │  int64  │
├──────────────────────────┼─────────────────────┼─────────────────────┼─────────┤
│ MzSiedKonars-PM2.5-1g    │ 2024-06-07 00:00:00 │ 2025-01-01 00:00:00 │    4993 │
│ PdBialUpalna-PM2.5-1g    │ 2024-01-01 01:00:00 │ 2024-01-25 11:00:00 │     587 │
│ PdGrajewoWPoMOB-PM2.5-1g │ 2024-12-09 12:00:00 │ 2025-01-01 00:00:00 │     541 │
│ KpWloclOkrze-PM2.5-1g    │ 2024-08-13 02:00:00 │ 2024-09-03 07:00:00 │     510 │
│ MzLegZegrzyn-PM2.5-1g    │ 2024-11-29 12:00:00 │ 2024-12-20 09:00:00 │     502 │
│ LdLodzCzerni-PM2.5-1g    │ 2024-04-28 11:00:00 │ 2024-05-18 21:00:00 │     491 │
│ SkPolaRuszcz-PM2.5-1g    │ 2024-09-11 02:00:00 │ 2024-09-30 20:00:00 │     475 │
│ PkRzeszSloci-PM2.5-1g    │ 2024-12-12 09:00:00 │ 2025-01-01 00:00:00 │     472 │
└───

Najdłuższa luka PM2.5 (`MzSiedKonars`, od 7 czerwca do końca roku) pokrywa się z datą zamknięcia stanowiska w metadanych (2024-06-06). Pozostałe długie luki trwają po 2-3 tygodnie.

### 3.7. Zawieszony czujnik

Przyrząd, który się zawiesił, zwraca przez wiele godzin tę samą liczbę. Stężenia pyłu zmieniają się z godziny na godzinę (mediana zmiany PM2.5 to ok. 1 µg/m³ - patrz 3.8), więc kilka godzin identycznej wartości jest podejrzane. Nie ma jednego obiektywnego progu, dlatego pokazujemy liczbę ciągów od 6, 12 i 24 godzin, osobno dla zer i dla innych wartości.

In [27]:
con.sql('''
SELECT
    wskaznik,
    wartosc = 0 AS zero,
    count(*) FILTER (WHERE godziny >= 6) AS od_6h,
    count(*) FILTER (WHERE godziny >= 12) AS od_12h,
    count(*) FILTER (WHERE godziny >= 24) AS od_24h,
    count(DISTINCT kod_stanowiska) AS stanowiska,
    max(godziny) AS najdluzszy_h
FROM ciagi
WHERE wartosc IS NOT NULL AND godziny >= 6 AND wskaznik IN ('PM10', 'PM2.5', 'NO2', 'NO', 'NOx', 'O3', 'SO2', 'CO', 'C6H6')
GROUP BY ALL
ORDER BY wskaznik, zero
''').show(max_rows=40, max_width=250)

┌──────────┬─────────┬───────┬────────┬────────┬────────────┬──────────────┐
│ wskaznik │  zero   │ od_6h │ od_12h │ od_24h │ stanowiska │ najdluzszy_h │
│ varchar  │ boolean │ int64 │ int64  │ int64  │   int64    │    int64     │
├──────────┼─────────┼───────┼────────┼────────┼────────────┼──────────────┤
│ C6H6     │ false   │  2346 │    361 │     50 │         43 │           73 │
│ C6H6     │ true    │    97 │     16 │      0 │          8 │           22 │
│ NO       │ false   │  2881 │    304 │      4 │         90 │           50 │
│ NO       │ true    │   428 │    134 │     35 │         26 │          120 │
│ NO2      │ false   │    22 │      0 │      0 │          7 │           10 │
│ NO2      │ true    │     1 │      0 │      0 │          1 │            6 │
│ NOx      │ false   │     2 │      1 │      1 │          2 │           25 │
│ NOx      │ true    │     1 │      0 │      0 │          1 │           11 │
│ O3       │ false   │     6 │      0 │      0 │          3 │            8 │

Dla gazów o niskich stężeniach (SO2, NO, benzen) długie ciągi stałej wartości mogą wynikać z dokładności zapisu: wartości są podawane z rozdzielczością 0,1 (SO2, NO) albo 0,01 (benzen), a typowy poziom jest bliski zera. Dla pyłu to mniej prawdopodobne. Najdłuższe ciągi dla PM:

In [28]:
con.sql('''
SELECT kod_stanowiska, wartosc, poczatek, koniec, godziny
FROM ciagi
WHERE wartosc IS NOT NULL AND wskaznik IN ('PM2.5', 'PM10') AND godziny >= 12
ORDER BY godziny DESC
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬─────────┬─────────────────────┬─────────────────────┬─────────┐
│      kod_stanowiska      │ wartosc │      poczatek       │       koniec        │ godziny │
│         varchar          │ double  │      timestamp      │      timestamp      │  int64  │
├──────────────────────────┼─────────┼─────────────────────┼─────────────────────┼─────────┤
│ PdBorsukowiz-PM10-1g     │     0.0 │ 2024-02-17 18:00:00 │ 2024-02-18 23:00:00 │      30 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-02-17 18:00:00 │ 2024-02-18 23:00:00 │      30 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-09-30 13:00:00 │ 2024-10-01 12:00:00 │      24 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-12-17 01:00:00 │ 2024-12-17 22:00:00 │      22 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-11-02 00:00:00 │ 2024-11-02 21:00:00 │      22 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-09-29 10:00:00 │ 2024-09-30 06:00:00 │      21 │
│ PdBorsukowiz-PM10-1g     │     0.0 │ 2024-12-17 01:00:00 │ 2024-12-1

Najdłuższe ciągi stałej wartości dla pyłu to prawie wyłącznie zera na stacji `PdBorsukowiz`, równocześnie na stanowisku PM2.5 i PM10 (np. 30 godzin 17-18 lutego). Ciągi niezerowe są krótsze (do 17 godzin) i dotyczą niskich wartości.

### 3.8. Skoki

Najpierw rozkład zmiany wartości między kolejnymi godzinami dla pyłu, żeby dobrać próg na podstawie danych:

In [29]:
con.sql('''
WITH zmiany AS (
    SELECT
        wskaznik,
        abs(wartosc - lag(wartosc) OVER (PARTITION BY kod_stanowiska ORDER BY data_czas)) AS zmiana
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
)
SELECT
    wskaznik,
    round(median(zmiana), 1) AS p50,
    round(quantile_cont(zmiana, 0.99), 1) AS p99,
    round(quantile_cont(zmiana, 0.999), 1) AS p99_9,
    round(quantile_cont(zmiana, 0.9999), 1) AS p99_99,
    max(zmiana) AS max
FROM zmiany
GROUP BY wskaznik
''').show(max_rows=40, max_width=250)

┌──────────┬────────┬────────┬────────┬────────┬────────┐
│ wskaznik │  p50   │  p99   │ p99_9  │ p99_99 │  max   │
│ varchar  │ double │ double │ double │ double │ double │
├──────────┼────────┼────────┼────────┼────────┼────────┤
│ PM2.5    │    1.0 │   18.6 │   48.0 │   97.8 │  322.3 │
│ PM10     │    2.1 │   28.4 │   69.8 │  139.8 │  324.3 │
└──────────┴────────┴────────┴────────┴────────┴────────┘



Pojedynczy skok w górę może być początkiem prawdziwego epizodu smogowego. Za pik uznajemy więc godzinę, która jest wyższa **zarówno od poprzedniej, jak i od następnej** o więcej niż X µg/m³ - wartość wyskakuje na jedną godzinę i od razu wraca. Progi 50 i 100 µg/m³ odpowiadają mniej więcej percentylom 99,9 i 99,99 zmian godzinowych PM2.5.

In [30]:
con.sql('''
WITH sasiedzi AS (
    SELECT
        wskaznik,
        wartosc,
        wartosc - lag(wartosc) OVER w AS wzrost,
        wartosc - lead(wartosc) OVER w AS spadek
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
    WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
)
SELECT
    wskaznik,
    count(wartosc) AS pomiary,
    count(*) FILTER (WHERE wzrost > 50 AND spadek > 50) AS piki_ponad_50,
    count(*) FILTER (WHERE wzrost > 100 AND spadek > 100) AS piki_ponad_100
FROM sasiedzi
GROUP BY wskaznik
''').show(max_rows=40, max_width=250)

┌──────────┬─────────┬───────────────┬────────────────┐
│ wskaznik │ pomiary │ piki_ponad_50 │ piki_ponad_100 │
│ varchar  │  int64  │     int64     │     int64      │
├──────────┼─────────┼───────────────┼────────────────┤
│ PM10     │ 1519684 │           530 │             83 │
│ PM2.5    │  817648 │            91 │             14 │
└──────────┴─────────┴───────────────┴────────────────┘



In [31]:
con.sql('''
WITH sasiedzi AS (
    SELECT
        kod_stanowiska,
        data_czas,
        lag(wartosc) OVER w AS poprzednia,
        wartosc,
        lead(wartosc) OVER w AS nastepna
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g' AND wskaznik = 'PM2.5'
    WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
)
SELECT *
FROM sasiedzi
WHERE wartosc - poprzednia > 50 AND wartosc - nastepna > 50
ORDER BY wartosc - greatest(poprzednia, nastepna) DESC
LIMIT 10
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬─────────────────────┬────────────┬─────────┬──────────┐
│      kod_stanowiska      │      data_czas      │ poprzednia │ wartosc │ nastepna │
│         varchar          │      timestamp      │   double   │ double  │  double  │
├──────────────────────────┼─────────────────────┼────────────┼─────────┼──────────┤
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-03-09 18:00:00 │       42.7 │   365.0 │     67.0 │
│ DsWrocAlWisn-PM2.5-1g    │ 2024-03-31 15:00:00 │       46.5 │   260.4 │     65.1 │
│ SkStaraZlota-PM2.5-1g    │ 2024-01-10 11:00:00 │      134.0 │   296.9 │     37.6 │
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-03-25 21:00:00 │       96.0 │   268.4 │    126.7 │
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-07-23 11:00:00 │        4.5 │   141.5 │      2.7 │
│ DsKudowaSzkoMOB-PM2.5-1g │ 2024-11-07 21:00:00 │       89.9 │   220.3 │     43.4 │
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-07-22 19:00:00 │       14.8 │   177.8 │     54.6 │
│ SkSkarz1MajaMOB-PM2.5-1g │ 2024-01-16 18:00:00 │       47.9 │  

Piki PM2.5 powtarzają się na tych samych stanowiskach, najczęściej na `WmDzialPolnaMOB` (stacja mobilna): 19 z 91 pików ponad 50 µg/m³. Skrajny przypadek: 42,7 → 365,0 → 67,0 µg/m³ w ciągu trzech godzin.

## 4. Profil metadanych (`stacje`, `stanowiska`)

Metadane to słowniki: opisują stacje i stanowiska, do których odwołują się pomiary. Ich błędy nie psują samych pomiarów, ale psują złączenia - np. pomiar ze stanowiska, którego nie ma w słowniku, nie dostanie województwa ani współrzędnych.

### 4.1. Liczba rekordów i braki

In [32]:
con.sql('''
SELECT
    'stacje' AS tabela,
    count(*) AS rekordy,
    count(DISTINCT kod_stacji) AS unikalne_kody
FROM stacje
UNION ALL
SELECT
    'stanowiska',
    count(*),
    count(DISTINCT kod_stanowiska)
FROM stanowiska
''').show(max_rows=40, max_width=250)

┌────────────┬─────────┬───────────────┐
│   tabela   │ rekordy │ unikalne_kody │
│  varchar   │  int64  │     int64     │
├────────────┼─────────┼───────────────┤
│ stacje     │    1148 │          1147 │
│ stanowiska │    5849 │          5843 │
└────────────┴─────────┴───────────────┘



Liczba `NULL` w każdej kolumnie. `COLUMNS(*)` to skrót DuckDB: stosuje wyrażenie `count(*) - count(...)` do każdej kolumny tabeli osobno, więc nie trzeba wypisywać ich ręcznie.

In [33]:
con.sql('''
SELECT count(*) - count(COLUMNS(*)) FROM stacje
''').show(max_rows=40, max_width=250)

┌────────────┬────────────────────┬──────────────┬──────────────────┬───────────────────┬─────────────────┬────────────┬─────────────┬───────────────┬─────────────┬─────────────┬───────┬───────────┬─────────┐
│ kod_stacji │ kod_miedzynarodowy │ nazwa_stacji │ stary_kod_stacji │ data_uruchomienia │ data_zamkniecia │ typ_stacji │ typ_obszaru │ rodzaj_stacji │ wojewodztwo │ miejscowosc │ adres │ szerokosc │ dlugosc │
│   int64    │       int64        │    int64     │      int64       │       int64       │      int64      │   int64    │    int64    │     int64     │    int64    │    int64    │ int64 │   int64   │  int64  │
├────────────┼────────────────────┼──────────────┼──────────────────┼───────────────────┼─────────────────┼────────────┼─────────────┼───────────────┼─────────────┼─────────────┼───────┼───────────┼─────────┤
│          0 │                351 │            0 │              852 │                 0 │             275 │          0 │           0 │             0 │           0 │

In [34]:
con.sql('''
SELECT count(*) - count(COLUMNS(*)) FROM stanowiska
''').show(max_rows=40, max_width=250)

┌────────────────┬────────────┬──────────┬────────────────┬──────────────────┬─────────────┬───────────────────┬─────────────────┬─────────────┬────────┐
│ kod_stanowiska │ kod_stacji │ wskaznik │ wskaznik_nazwa │ czas_usredniania │ typ_pomiaru │ data_uruchomienia │ data_zamkniecia │ wojewodztwo │ strefa │
│     int64      │   int64    │  int64   │     int64      │      int64       │    int64    │       int64       │      int64      │    int64    │ int64  │
├────────────────┼────────────┼──────────┼────────────────┼──────────────────┼─────────────┼───────────────────┼─────────────────┼─────────────┼────────┤
│              0 │          0 │        0 │              0 │                0 │           0 │                 0 │            1800 │           0 │      0 │
└────────────────┴────────────┴──────────┴────────────────┴──────────────────┴─────────────┴───────────────────┴─────────────────┴─────────────┴────────┘



Większość braków jest oczekiwana: nie każda stacja ma kod międzynarodowy, poprzedni kod czy adres, a pusta data zamknięcia oznacza stację albo stanowisko, które nadal działa. Wśród 1800 pustych dat zamknięcia stanowisk są 4 daty `31.12.20025` (literówka w roku), których nie dało się odczytać przy konwersji do bronze (notebook 02, kontrola spójności).

### 4.2. Wartości zastępcze

Zamiast pustego pola źródło czasem wpisuje tekst `BRAK DANYCH` albo liczbę spoza zakresu. Taka wartość nie jest `NULL`, więc zwykłe liczenie braków jej nie wychwyci. `UNPIVOT` zamienia kolumny tekstowe na pary (kolumna, wartość), dzięki czemu jednym zapytaniem przeszukujemy wszystkie.

In [35]:
con.sql('''
SELECT 'stacje' AS tabela, kolumna, count(*) AS brak_danych
FROM (
    UNPIVOT (SELECT * EXCLUDE (data_uruchomienia, data_zamkniecia, szerokosc, dlugosc) FROM stacje)
    ON COLUMNS(*) INTO NAME kolumna VALUE wartosc
)
WHERE wartosc = 'BRAK DANYCH'
GROUP BY kolumna
UNION ALL
SELECT 'stanowiska', kolumna, count(*)
FROM (
    UNPIVOT (SELECT * EXCLUDE (data_uruchomienia, data_zamkniecia) FROM stanowiska)
    ON COLUMNS(*) INTO NAME kolumna VALUE wartosc
)
WHERE wartosc = 'BRAK DANYCH'
GROUP BY kolumna
''').show(max_rows=40, max_width=250)

┌─────────┬───────────────┬─────────────┐
│ tabela  │    kolumna    │ brak_danych │
│ varchar │    varchar    │    int64    │
├─────────┼───────────────┼─────────────┤
│ stacje  │ typ_stacji    │           4 │
│ stacje  │ rodzaj_stacji │          21 │
└─────────┴───────────────┴─────────────┘



Współrzędne mają własną wartość zastępczą - `-999` zamiast braku:

In [36]:
con.sql('''
SELECT
    count(*) FILTER (WHERE szerokosc = -999 OR dlugosc = -999) AS wspolrzedne_minus_999,
    count(*) FILTER (
        WHERE (szerokosc = -999 OR dlugosc = -999)
        AND (data_zamkniecia IS NULL OR data_zamkniecia >= DATE '2024-01-01')
    ) AS w_tym_aktywne_w_2024,
    count(*) FILTER (
        WHERE (szerokosc = -999 OR dlugosc = -999)
        AND kod_stacji IN (SELECT kod_stacji FROM pomiary_2024)
    ) AS w_tym_z_pomiarami_2024
FROM stacje
''').show(max_rows=40, max_width=250)

┌───────────────────────┬──────────────────────┬────────────────────────┐
│ wspolrzedne_minus_999 │ w_tym_aktywne_w_2024 │ w_tym_z_pomiarami_2024 │
│         int64         │        int64         │         int64          │
├───────────────────────┼──────────────────────┼────────────────────────┤
│                    76 │                    0 │                      0 │
└───────────────────────┴──────────────────────┴────────────────────────┘


Czy stacje z wartością `BRAK DANYCH` mają pomiary w 2024 r.?

In [37]:
con.sql('''
SELECT kod_stacji, typ_stacji, rodzaj_stacji, data_zamkniecia
FROM stacje
WHERE (typ_stacji = 'BRAK DANYCH' OR rodzaj_stacji = 'BRAK DANYCH')
    AND kod_stacji IN (SELECT kod_stacji FROM pomiary_2024)
''').show(max_rows=40, max_width=250)

┌────────────┬────────────┬───────────────┬─────────────────┐
│ kod_stacji │ typ_stacji │ rodzaj_stacji │ data_zamkniecia │
│  varchar   │  varchar   │    varchar    │      date       │
└────────────┴────────────┴───────────────┴─────────────────┘
                           0 rows                          


### 4.3. Współrzędne

Po pominięciu `-999` sprawdzamy, czy współrzędne leżą w prostokącie obejmującym Polskę (szerokość 49,0-54,9° N, długość 14,1-24,2° E).

In [38]:
con.sql('''
SELECT
    count(*) AS stacje,
    min(szerokosc) AS szerokosc_min,
    max(szerokosc) AS szerokosc_max,
    min(dlugosc) AS dlugosc_min,
    max(dlugosc) AS dlugosc_max,
    count(*) FILTER (
        WHERE szerokosc NOT BETWEEN 49.0 AND 54.9 OR dlugosc NOT BETWEEN 14.1 AND 24.2
    ) AS poza_polska
FROM stacje
WHERE szerokosc <> -999 AND dlugosc <> -999
''').show(max_rows=40, max_width=250)

┌────────┬───────────────┬───────────────┬─────────────┬─────────────┬─────────────┐
│ stacje │ szerokosc_min │ szerokosc_max │ dlugosc_min │ dlugosc_max │ poza_polska │
│ int64  │    double     │    double     │   double    │   double    │    int64    │
├────────┼───────────────┼───────────────┼─────────────┼─────────────┼─────────────┤
│   1072 │     49.293564 │     54.793639 │    14.23995 │   23.913056 │           0 │
└────────┴───────────────┴───────────────┴─────────────┴─────────────┴─────────────┘



### 4.4. Duplikaty i sprzeczności

Kody stacji i stanowisk powinny być unikalne. Zdublowane kody:

In [39]:
con.sql('''
SELECT *
FROM stacje
WHERE kod_stacji IN (SELECT kod_stacji FROM stacje GROUP BY kod_stacji HAVING count(*) > 1)
''').show(max_rows=40, max_width=250)

┌─────────────┬────────────────────┬────────────────────────┬──────────────────┬───────────────────┬─────────────────┬────────────┬─────────────┬───────────────┬─────────────┬─────────────┬───────────────┬───────────┬──────────┐
│ kod_stacji  │ kod_miedzynarodowy │      nazwa_stacji      │ stary_kod_stacji │ data_uruchomienia │ data_zamkniecia │ typ_stacji │ typ_obszaru │ rodzaj_stacji │ wojewodztwo │ miejscowosc │     adres     │ szerokosc │ dlugosc  │
│   varchar   │      varchar       │        varchar         │     varchar      │       date        │      date       │  varchar   │   varchar   │    varchar    │   varchar   │   varchar   │    varchar    │  double   │  double  │
├─────────────┼────────────────────┼────────────────────────┼──────────────────┼───────────────────┼─────────────────┼────────────┼─────────────┼───────────────┼─────────────┼─────────────┼───────────────┼───────────┼──────────┤
│ OpKrap3Maja │ PL0810A            │ Krapkowice, ul. 3 Maja │ NULL             │ 202

In [40]:
con.sql('''
SELECT
    kod_stanowiska,
    count(*) AS wiersze,
    count(DISTINCT stanowiska) AS rozne_wiersze
FROM stanowiska
GROUP BY kod_stanowiska
HAVING count(*) > 1
ORDER BY kod_stanowiska
''').show(max_rows=40, max_width=250)

┌──────────────────────────────┬─────────┬───────────────┐
│        kod_stanowiska        │ wiersze │ rozne_wiersze │
│           varchar            │  int64  │     int64     │
├──────────────────────────────┼─────────┼───────────────┤
│ PdSuwPulaski-BaA(PM10)-24g   │       2 │             1 │
│ PdSuwPulaski-BbF(PM10)-24g   │       2 │             1 │
│ PdSuwPulaski-BjF(PM10)-24g   │       2 │             1 │
│ PdSuwPulaski-BkF(PM10)-24g   │       2 │             1 │
│ PdSuwPulaski-DBahA(PM10)-24g │       2 │             1 │
│ PdSuwPulaski-IP(PM10)-24g    │       2 │             1 │
└──────────────────────────────┴─────────┴───────────────┘



`count(DISTINCT stanowiska)` porównuje całe wiersze: wartość 1 oznacza, że oba wiersze są identyczne we wszystkich kolumnach. Zdublowana stacja różni się tylko kolumną `stary_kod_stacji`.

Sprzeczne daty i kody:

In [41]:
con.sql('''
SELECT 'stacje: zamknięcie przed uruchomieniem' AS sprawdzenie, count(*) AS liczba
FROM stacje
WHERE data_zamkniecia < data_uruchomienia
UNION ALL
SELECT 'stanowiska: zamknięcie przed uruchomieniem', count(*)
FROM stanowiska
WHERE data_zamkniecia < data_uruchomienia
UNION ALL
SELECT 'stacje: stary kod równy obecnemu', count(*)
FROM stacje
WHERE stary_kod_stacji = kod_stacji
''').show(max_rows=40, max_width=250)

┌────────────────────────────────────────────┬────────┐
│                sprawdzenie                 │ liczba │
│                  varchar                   │ int64  │
├────────────────────────────────────────────┼────────┤
│ stacje: zamknięcie przed uruchomieniem     │      1 │
│ stanowiska: zamknięcie przed uruchomieniem │      0 │
│ stacje: stary kod równy obecnemu           │      1 │
└────────────────────────────────────────────┴────────┘



In [42]:
con.sql('''
SELECT kod_stacji, stary_kod_stacji, data_uruchomienia, data_zamkniecia
FROM stacje
WHERE data_zamkniecia < data_uruchomienia OR stary_kod_stacji = kod_stacji
''').show(max_rows=40, max_width=250)

┌──────────────┬──────────────────┬───────────────────┬─────────────────┐
│  kod_stacji  │ stary_kod_stacji │ data_uruchomienia │ data_zamkniecia │
│   varchar    │     varchar      │       date        │      date       │
├──────────────┼──────────────────┼───────────────────┼─────────────────┤
│ OpNysaRodzi2 │ OpNysaRodzie     │ 2025-01-01        │ 2024-12-31      │
│ SlZabSkloCu2 │ SlZabSkloCu2     │ 2024-01-01        │ NULL            │
└──────────────┴──────────────────┴───────────────────┴─────────────────┘



### 4.5. Spójność kluczy z pomiarami

Każdy kod z `pomiary_2024` powinien mieć wpis w metadanych. `ANTI JOIN` zwraca wiersze z lewej tabeli, które nie mają pary w prawej.

In [43]:
con.sql('''
SELECT 'stacje z pomiarów bez wpisu w stacje' AS sprawdzenie, count(*) AS liczba
FROM (SELECT DISTINCT kod_stacji FROM pomiary_2024) AS p
ANTI JOIN stacje USING (kod_stacji)
UNION ALL
SELECT 'stanowiska z pomiarów bez wpisu w stanowiska', count(*)
FROM (SELECT DISTINCT kod_stanowiska FROM pomiary_2024) AS p
ANTI JOIN stanowiska USING (kod_stanowiska)
UNION ALL
SELECT 'stanowiska (wiersze) wskazujące stację spoza stacje', count(*)
FROM stanowiska
ANTI JOIN stacje USING (kod_stacji)
''').show(max_rows=40, max_width=250)

┌─────────────────────────────────────────────────────┬────────┐
│                     sprawdzenie                     │ liczba │
│                       varchar                       │ int64  │
├─────────────────────────────────────────────────────┼────────┤
│ stacje z pomiarów bez wpisu w stacje                │      2 │
│ stanowiska z pomiarów bez wpisu w stanowiska        │     17 │
│ stanowiska (wiersze) wskazujące stację spoza stacje │      6 │
└─────────────────────────────────────────────────────┴────────┘



Stanowiska z pomiarami, których nie ma w słowniku stanowisk:

In [44]:
con.sql('''
SELECT
    p.kod_stanowiska,
    p.kod_stacji,
    p.kod_stacji IN (SELECT kod_stacji FROM stacje) AS stacja_w_metadanych,
    count(p.wartosc) AS pomiary
FROM pomiary_2024 AS p
ANTI JOIN stanowiska USING (kod_stanowiska)
GROUP BY p.kod_stanowiska, p.kod_stacji
ORDER BY p.kod_stanowiska
''').show(max_rows=40, max_width=250)

┌─────────────────────────┬─────────────────┬─────────────────────┬─────────┐
│     kod_stanowiska      │   kod_stacji    │ stacja_w_metadanych │ pomiary │
│         varchar         │     varchar     │       boolean       │  int64  │
├─────────────────────────┼─────────────────┼─────────────────────┼─────────┤
│ DsStrzegomMOB-NO-1g     │ DsStrzegomMOB   │ true                │    7826 │
│ KpSwiecJPawlMOB-NO-1g   │ KpSwiecJPawlMOB │ true                │    8662 │
│ KpZielBoryTu-2penten-1g │ KpZielBoryTu    │ true                │     291 │
│ LdLodzCzerni-NO-1g      │ LdLodzCzerni    │ true                │    8758 │
│ LuZielWyszyn-PM10-1g    │ LuZielWyszyn    │ true                │    8761 │
│ OpBrzegPoprzMOB-PM10-1g │ OpBrzegPoprzMOB │ false               │    8340 │
│ PdBialAlPils-NO-1g      │ PdBialAlPils    │ true                │    8764 │
│ PdBialPPiech-NO-1g      │ PdBialPPiech    │ true                │    8702 │
│ PdLomSikorsk-C6H6-1g    │ PdLomSikorsk    │ true              

Z 17 brakujących stanowisk 8 to tlenek azotu `NO` - na tych 8 stacjach słownik zawiera stanowiska `NO2` i `NOx`, ale nie `NO`. Tylko 2 z 17 należą do stacji, których kodu nie ma w słowniku stacji. Dla nich szukamy w słowniku najbardziej podobnego kodu (`jaro_winkler_similarity` zwraca 1 dla identycznych napisów i mniej, im bardziej się różnią):

W drugą stronę: ile stacji według metadanych działało w 2024 r., a ile ma pomiary?

In [45]:
con.sql('''
SELECT
    p.kod_stacji AS kod_w_pomiarach,
    s.kod_stacji AS najpodobniejszy_w_stacje,
    round(jaro_winkler_similarity(p.kod_stacji, s.kod_stacji), 3) AS podobienstwo,
    s.nazwa_stacji,
    s.data_zamkniecia
FROM (SELECT DISTINCT kod_stacji FROM pomiary_2024 ANTI JOIN stacje USING (kod_stacji)) AS p
CROSS JOIN stacje AS s
QUALIFY row_number() OVER (
    PARTITION BY p.kod_stacji
    ORDER BY jaro_winkler_similarity(p.kod_stacji, s.kod_stacji) DESC
) = 1
''').show(max_rows=40, max_width=250)

┌─────────────────┬──────────────────────────┬──────────────┬───────────────────────┬─────────────────┐
│ kod_w_pomiarach │ najpodobniejszy_w_stacje │ podobienstwo │     nazwa_stacji      │ data_zamkniecia │
│     varchar     │         varchar          │    double    │        varchar        │      date       │
├─────────────────┼──────────────────────────┼──────────────┼───────────────────────┼─────────────────┤
│ ZpSzczLaczna    │ ZpSzczLacz04             │        0.933 │ Szczecin_Łączna       │ NULL            │
│ OpBrzegPoprzMOB │ OpBrzegPoprz             │         0.96 │ Brzeg, ul. Poprzeczna │ NULL            │
└─────────────────┴──────────────────────────┴──────────────┴───────────────────────┴─────────────────┘



Obie stacje mają w metadanych wpis pod innym kodem (`ZpSzczLacz04` - Szczecin, ul. Łączna; `OpBrzegPoprz` - Brzeg, ul. Poprzeczna). Kod w pliku pomiarów i kod w słowniku nie są więc zsynchronizowane - złączenie po `kod_stacji` zgubi te stacje.

W drugą stronę: ile stacji według metadanych działało w 2024 r., a ile ma pomiary?

In [46]:
con.sql('''
WITH aktywne AS (
    SELECT DISTINCT kod_stacji
    FROM stacje
    WHERE data_uruchomienia <= DATE '2024-12-31'
        AND (data_zamkniecia IS NULL OR data_zamkniecia >= DATE '2024-01-01')
)
SELECT
    count(*) AS aktywne_w_2024,
    count(*) FILTER (WHERE kod_stacji IN (SELECT kod_stacji FROM pomiary_2024)) AS z_pomiarami,
    count(*) FILTER (WHERE kod_stacji NOT IN (SELECT kod_stacji FROM pomiary_2024)) AS bez_pomiarow
FROM aktywne
''').show(max_rows=40, max_width=250)

┌────────────────┬─────────────┬──────────────┐
│ aktywne_w_2024 │ z_pomiarami │ bez_pomiarow │
│     int64      │    int64    │    int64     │
├────────────────┼─────────────┼──────────────┤
│            292 │         284 │            8 │
└────────────────┴─────────────┴──────────────┘



Lista stacji aktywnych według metadanych, które nie mają pomiarów w 2024 r.:

In [47]:
con.sql('''
WITH aktywne AS (
    SELECT DISTINCT kod_stacji
    FROM stacje
    WHERE data_uruchomienia <= DATE '2024-12-31'
        AND (data_zamkniecia IS NULL OR data_zamkniecia >= DATE '2024-01-01')
)
SELECT
    t.kod_stacji,
    any_value(t.data_uruchomienia) AS uruchomienie,
    any_value(t.data_zamkniecia) AS zamkniecie,
    count(s.kod_stanowiska) AS stanowiska_w_slowniku
FROM aktywne AS a
JOIN stacje AS t USING (kod_stacji)
LEFT JOIN stanowiska AS s USING (kod_stacji)
WHERE a.kod_stacji NOT IN (SELECT kod_stacji FROM pomiary_2024)
GROUP BY t.kod_stacji
ORDER BY t.kod_stacji
''').show(max_rows=40, max_width=250)

┌─────────────────┬──────────────┬────────────┬───────────────────────┐
│   kod_stacji    │ uruchomienie │ zamkniecie │ stanowiska_w_slowniku │
│     varchar     │     date     │    date    │         int64         │
├─────────────────┼──────────────┼────────────┼───────────────────────┤
│ MpKrakDietla    │ 2016-01-01   │ 2024-01-31 │                     4 │
│ OpBrzegPoprz    │ 2021-01-11   │ NULL       │                    12 │
│ PmGdaPowWars    │ 1996-10-01   │ NULL       │                     7 │
│ PmUstkaRybacMOB │ 2023-01-01   │ 2024-01-31 │                     4 │
│ SlCzestoZana    │ 2010-01-01   │ NULL       │                     1 │
│ WpKrotoOlimpMOB │ 2024-12-19   │ NULL       │                     2 │
│ ZpSwinBialorMOB │ 2024-12-18   │ 2025-12-17 │                     4 │
│ ZpSzczLacz04    │ 2004-12-31   │ NULL       │                     6 │
└─────────────────┴──────────────┴────────────┴───────────────────────┘



Dwie z nich to stacje z poprzedniego zapytania pod kodem ze słownika (`OpBrzegPoprz`, `ZpSzczLacz04`). Cztery działały w 2024 r. tylko przez kilka tygodni: zamknięte 31 stycznia albo uruchomione w drugiej połowie grudnia. Dwie pozostałe (`PmGdaPowWars`, `SlCzestoZana`) według metadanych działają, ale nie mają żadnych pomiarów w archiwum 2024.

### 4.6. Pomiary poza okresem działania

Porównujemy datę pierwszego i ostatniego niepustego pomiaru każdego stanowiska z jego datami uruchomienia i zamknięcia. Dla pomiarów godzinowych odejmujemy godzinę, bo GIOŚ oznacza godzinę jej końcem (pomiar z `00:00` należy do poprzedniego dnia).

In [48]:
con.sql('''
WITH okres AS (
    SELECT
        kod_stanowiska,
        any_value(data_uruchomienia) AS uruchomienie,
        any_value(data_zamkniecia) AS zamkniecie
    FROM stanowiska
    GROUP BY kod_stanowiska
),
dane AS (
    SELECT
        kod_stanowiska,
        min(CASE WHEN czas_usredniania = '1g' THEN data_czas - INTERVAL 1 HOUR ELSE data_czas END)::DATE AS pierwszy_pomiar,
        max(CASE WHEN czas_usredniania = '1g' THEN data_czas - INTERVAL 1 HOUR ELSE data_czas END)::DATE AS ostatni_pomiar
    FROM pomiary_2024
    WHERE wartosc IS NOT NULL
    GROUP BY kod_stanowiska
)
SELECT
    count(*) AS stanowiska,
    count(*) FILTER (WHERE zamkniecie < DATE '2024-01-01') AS zamkniete_przed_2024,
    count(*) FILTER (WHERE pierwszy_pomiar < uruchomienie) AS pomiar_przed_uruchomieniem,
    count(*) FILTER (WHERE ostatni_pomiar > zamkniecie) AS pomiar_po_zamknieciu
FROM dane
JOIN okres USING (kod_stanowiska)
''').show(max_rows=40, max_width=250)

┌────────────┬──────────────────────┬────────────────────────────┬──────────────────────┐
│ stanowiska │ zamkniete_przed_2024 │ pomiar_przed_uruchomieniem │ pomiar_po_zamknieciu │
│   int64    │        int64         │           int64            │        int64         │
├────────────┼──────────────────────┼────────────────────────────┼──────────────────────┤
│       1742 │                    0 │                          4 │                    5 │
└────────────┴──────────────────────┴────────────────────────────┴──────────────────────┘



In [49]:
con.sql('''
WITH okres AS (
    SELECT
        kod_stanowiska,
        any_value(data_uruchomienia) AS uruchomienie,
        any_value(data_zamkniecia) AS zamkniecie
    FROM stanowiska
    GROUP BY kod_stanowiska
),
dane AS (
    SELECT
        kod_stanowiska,
        min(CASE WHEN czas_usredniania = '1g' THEN data_czas - INTERVAL 1 HOUR ELSE data_czas END)::DATE AS pierwszy_pomiar,
        max(CASE WHEN czas_usredniania = '1g' THEN data_czas - INTERVAL 1 HOUR ELSE data_czas END)::DATE AS ostatni_pomiar
    FROM pomiary_2024
    WHERE wartosc IS NOT NULL
    GROUP BY kod_stanowiska
)
SELECT
    kod_stanowiska,
    uruchomienie,
    pierwszy_pomiar,
    ostatni_pomiar,
    zamkniecie,
    greatest(uruchomienie - pierwszy_pomiar, ostatni_pomiar - zamkniecie) AS dni_poza_okresem
FROM dane
JOIN okres USING (kod_stanowiska)
WHERE pierwszy_pomiar < uruchomienie OR ostatni_pomiar > zamkniecie
ORDER BY dni_poza_okresem DESC
''').show(max_rows=40, max_width=250)

┌───────────────────────────────┬──────────────┬─────────────────┬────────────────┬────────────┬──────────────────┐
│        kod_stanowiska         │ uruchomienie │ pierwszy_pomiar │ ostatni_pomiar │ zamkniecie │ dni_poza_okresem │
│            varchar            │     date     │      date       │      date      │    date    │      int64       │
├───────────────────────────────┼──────────────┼─────────────────┼────────────────┼────────────┼──────────────────┤
│ MpMuszynLodoMOB-BaP(PM10)-24g │ 2023-01-01   │ 2024-01-01      │ 2024-12-31     │ 2024-12-12 │               19 │
│ OpStrzOpWyszMOB-PM10-1g       │ 2024-01-05   │ 2024-01-05      │ 2024-12-30     │ 2024-12-19 │               11 │
│ OpStrzOpWyszMOB-PM2.5-1g      │ 2024-01-05   │ 2024-01-05      │ 2024-12-30     │ 2024-12-19 │               11 │
│ WpKoscianMayMOB-PM10-1g       │ 2023-02-16   │ 2024-01-01      │ 2024-12-23     │ 2024-12-18 │                5 │
│ WpKoscianMayMOB-PM2.5-1g      │ 2023-02-16   │ 2024-01-01      │ 2024-

Żadne stanowisko zamknięte przed 2024 r. nie ma pomiarów. 9 stanowisk ma pomiary poza okresem z metadanych i wszystkie należą do stacji mobilnych (`MOB`): 5 ma pomiary od 5 do 19 dni po dacie zamknięcia ze słownika, a 4 (`LuLubsStrzelMOB`) - pomiar z dnia przed datą uruchomienia.

## 5. Profil pogody (`openmeteo_2024`)

Pogoda to dane z modelu meteorologicznego, a nie pomiar na stacji GIOŚ. Sprawdzamy kompletność, zakresy fizyczne oraz to, jak punkt modelu ma się do położenia stacji.

### 5.1. Liczba rekordów i siatka czasu

In [50]:
con.sql('''
SELECT
    count(*) AS rekordy,
    count(DISTINCT kod_stacji) AS stacje,
    count(DISTINCT data_czas_utc) AS godziny,
    count(*) - count(DISTINCT (kod_stacji, data_czas_utc)) AS duplikaty,
    min(data_czas_utc) AS pierwsza,
    max(data_czas_utc) AS ostatnia
FROM openmeteo_2024
''').show(max_rows=40, max_width=250)

┌─────────┬────────┬─────────┬───────────┬─────────────────────┬─────────────────────┐
│ rekordy │ stacje │ godziny │ duplikaty │      pierwsza       │      ostatnia       │
│  int64  │ int64  │  int64  │   int64   │      timestamp      │      timestamp      │
├─────────┼────────┼─────────┼───────────┼─────────────────────┼─────────────────────┤
│  843264 │     96 │    8784 │         0 │ 2024-01-01 00:00:00 │ 2024-12-31 23:00:00 │
└─────────┴────────┴─────────┴───────────┴─────────────────────┴─────────────────────┘



96 stacji × 8784 godziny = 843 264 wiersze, bez duplikatów. Czas jest w UTC i oznacza początek godziny (`00:00`-`23:00`), a GIOŚ oznacza godzinę jej końcem w czasie bez strefy (3.6). Zgodność obu zapisów sprawdzamy przy porównaniu źródeł.

### 5.2. Braki i zakresy wartości

`UNPIVOT` zamienia 6 kolumn pogodowych na pary (zmienna, wartość) - tę samą operację wykonaliśmy w bronze na plikach GIOŚ. `INCLUDE NULLS` zostawia puste wartości, żeby dało się je policzyć.

In [51]:
con.sql('''
SELECT
    zmienna,
    count(*) - count(wartosc) AS braki,
    min(wartosc) AS min,
    round(avg(wartosc), 2) AS srednia,
    max(wartosc) AS max
FROM (
    SELECT *
    FROM openmeteo_2024
    UNPIVOT INCLUDE NULLS (
        wartosc FOR zmienna IN (
            temperature_2m, relative_humidity_2m, precipitation,
            surface_pressure, wind_speed_10m, wind_direction_10m
        )
    )
)
GROUP BY zmienna
ORDER BY zmienna
''').show(max_rows=40, max_width=250)

┌──────────────────────┬───────┬────────┬─────────┬────────┐
│       zmienna        │ braki │  min   │ srednia │  max   │
│       varchar        │ int64 │ double │ double  │ double │
├──────────────────────┼───────┼────────┼─────────┼────────┤
│ precipitation        │     0 │    0.0 │    0.08 │   20.7 │
│ relative_humidity_2m │     0 │   17.0 │   75.96 │  100.0 │
│ surface_pressure     │     0 │  942.1 │  996.99 │ 1038.2 │
│ temperature_2m       │     0 │  -21.5 │   11.14 │   34.9 │
│ wind_direction_10m   │     0 │    0.0 │  193.26 │  360.0 │
│ wind_speed_10m       │     0 │    0.0 │   12.89 │   54.5 │
└──────────────────────┴───────┴────────┴─────────┴────────┘



### 5.3. Wartości niemożliwe

Reguły wynikające z definicji zmiennych: wilgotność względna w zakresie 0-100%, opad i prędkość wiatru nieujemne, kierunek wiatru w zakresie 0-360°. Temperaturę porównujemy z szerokim zakresem -40 do 45 °C.

In [52]:
con.sql('''
SELECT
    count(*) FILTER (WHERE relative_humidity_2m < 0 OR relative_humidity_2m > 100) AS wilgotnosc_poza_0_100,
    count(*) FILTER (WHERE precipitation < 0) AS opad_ujemny,
    count(*) FILTER (WHERE wind_speed_10m < 0) AS wiatr_ujemny,
    count(*) FILTER (WHERE wind_direction_10m < 0 OR wind_direction_10m > 360) AS kierunek_poza_0_360,
    count(*) FILTER (WHERE temperature_2m < -40 OR temperature_2m > 45) AS temperatura_poza_zakresem
FROM openmeteo_2024
''').show(max_rows=40, max_width=250)

┌───────────────────────┬─────────────┬──────────────┬─────────────────────┬───────────────────────────┐
│ wilgotnosc_poza_0_100 │ opad_ujemny │ wiatr_ujemny │ kierunek_poza_0_360 │ temperatura_poza_zakresem │
│         int64         │    int64    │    int64     │        int64        │           int64           │
├───────────────────────┼─────────────┼──────────────┼─────────────────────┼───────────────────────────┤
│                     0 │           0 │            0 │                   0 │                         0 │
└───────────────────────┴─────────────┴──────────────┴─────────────────────┴───────────────────────────┘



**Ciśnienie.** `surface_pressure` to ciśnienie na poziomie terenu, więc zależy od wysokości: na stacjach położonych wyżej jest niższe. Żeby ocenić, czy wartości są rozsądne, przeliczamy je na poziom morza wzorem barometrycznym (z wysokością `wysokosc_m` i temperaturą z tej samej godziny) i porównujemy z typowym zakresem ok. 950-1050 hPa.

In [53]:
con.sql('''
WITH cisnienie AS (
    SELECT
        surface_pressure,
        surface_pressure * pow(
            1 - 0.0065 * wysokosc_m / (temperature_2m + 0.0065 * wysokosc_m + 273.15),
            -5.257
        ) AS cisnienie_morze
    FROM openmeteo_2024
)
SELECT
    round(min(surface_pressure), 1) AS teren_min,
    round(max(surface_pressure), 1) AS teren_max,
    round(min(cisnienie_morze), 1) AS morze_min,
    round(max(cisnienie_morze), 1) AS morze_max,
    count(*) FILTER (WHERE cisnienie_morze NOT BETWEEN 950 AND 1050) AS morze_poza_950_1050
FROM cisnienie
''').show(max_rows=40, max_width=250)

┌───────────┬───────────┬───────────┬───────────┬─────────────────────┐
│ teren_min │ teren_max │ morze_min │ morze_max │ morze_poza_950_1050 │
│  double   │  double   │  double   │  double   │        int64        │
├───────────┼───────────┼───────────┼───────────┼─────────────────────┤
│     942.1 │    1038.2 │     976.1 │    1041.7 │                   0 │
└───────────┴───────────┴───────────┴───────────┴─────────────────────┘



**Kodowanie kierunku wiatru.** Kierunek 0° i 360° to ta sama północ. Przy ciszy (prędkość 0) kierunek nie ma sensu, więc sprawdzamy, jaka wartość jest wtedy zapisana.

In [54]:
con.sql('''
SELECT
    count(*) FILTER (WHERE wind_direction_10m = 0) AS kierunek_0,
    count(*) FILTER (WHERE wind_direction_10m = 360) AS kierunek_360,
    count(*) FILTER (WHERE wind_speed_10m = 0) AS cisza,
    list(DISTINCT wind_direction_10m) FILTER (WHERE wind_speed_10m = 0) AS kierunek_przy_ciszy
FROM openmeteo_2024
''').show(max_rows=40, max_width=250)

┌────────────┬──────────────┬───────┬─────────────────────┐
│ kierunek_0 │ kierunek_360 │ cisza │ kierunek_przy_ciszy │
│   int64    │    int64     │ int64 │      double[]       │
├────────────┼──────────────┼───────┼─────────────────────┤
│          1 │         2854 │   106 │ [270.0]             │
└────────────┴──────────────┴───────┴─────────────────────┘



Północ jest zapisywana prawie zawsze jako 360°, a 0° pojawia się raz. Przy ciszy kierunek ma zawsze wartość 270°, czyli stałą wartość, a nie pomiar. W cechach modelu kierunek trzeba będzie zakodować tak, żeby 0° i 360° były tym samym (np. sinus i cosinus kąta), a cisze potraktować osobno.

### 5.4. Punkt siatki a położenie stacji

Według dokumentacji Open-Meteo `latitude` i `longitude` w odpowiedzi to środek komórki siatki modelu, z której pochodzi pogoda, a `elevation` to wysokość terenu z cyfrowego modelu terenu (rozdzielczość 90 m) dla współrzędnych z zapytania, czyli dla położenia stacji. W bronze to kolumny `siatka_szerokosc`, `siatka_dlugosc` i `wysokosc_m`.

Odległość między punktem siatki a współrzędnymi stacji z metadanych GIOŚ liczymy wzorem haversine (odległość po kuli o promieniu 6371 km):

In [55]:
con.sql('''
WITH siatka AS (
    SELECT
        kod_stacji,
        any_value(siatka_szerokosc) AS siatka_szerokosc,
        any_value(siatka_dlugosc) AS siatka_dlugosc,
        any_value(wysokosc_m) AS wysokosc_m
    FROM openmeteo_2024
    GROUP BY kod_stacji
),
wspolrzedne AS (
    SELECT kod_stacji, any_value(szerokosc) AS szerokosc, any_value(dlugosc) AS dlugosc
    FROM stacje
    GROUP BY kod_stacji
),
odleglosc AS (
    SELECT
        kod_stacji,
        wysokosc_m,
        2 * 6371 * asin(sqrt(
            sin(radians(siatka_szerokosc - szerokosc) / 2) ^ 2
            + cos(radians(szerokosc)) * cos(radians(siatka_szerokosc))
              * sin(radians(siatka_dlugosc - dlugosc) / 2) ^ 2
        )) AS km
    FROM siatka
    JOIN wspolrzedne USING (kod_stacji)
)

SELECT
    count(*) AS stacje,
    round(min(km), 2) AS min_km,
    round(median(km), 2) AS mediana_km,
    round(quantile_cont(km, 0.9), 2) AS p90_km,
    round(max(km), 2) AS max_km
FROM odleglosc
''').show(max_rows=40, max_width=250)

┌────────┬────────┬────────────┬────────┬────────┐
│ stacje │ min_km │ mediana_km │ p90_km │ max_km │
│ int64  │ double │   double   │ double │ double │
├────────┼────────┼────────────┼────────┼────────┤
│     96 │   0.15 │       3.79 │   6.01 │   8.35 │
└────────┴────────┴────────────┴────────┴────────┘



In [56]:
con.sql('''
WITH siatka AS (
    SELECT
        kod_stacji,
        any_value(siatka_szerokosc) AS siatka_szerokosc,
        any_value(siatka_dlugosc) AS siatka_dlugosc,
        any_value(wysokosc_m) AS wysokosc_m
    FROM openmeteo_2024
    GROUP BY kod_stacji
),
wspolrzedne AS (
    SELECT kod_stacji, any_value(szerokosc) AS szerokosc, any_value(dlugosc) AS dlugosc
    FROM stacje
    GROUP BY kod_stacji
),
odleglosc AS (
    SELECT
        kod_stacji,
        wysokosc_m,
        2 * 6371 * asin(sqrt(
            sin(radians(siatka_szerokosc - szerokosc) / 2) ^ 2
            + cos(radians(szerokosc)) * cos(radians(siatka_szerokosc))
              * sin(radians(siatka_dlugosc - dlugosc) / 2) ^ 2
        )) AS km
    FROM siatka
    JOIN wspolrzedne USING (kod_stacji)
)

SELECT kod_stacji, round(km, 2) AS km, wysokosc_m
FROM odleglosc
ORDER BY km DESC
LIMIT 5
''').show(max_rows=40, max_width=250)

┌──────────────┬────────┬────────────┐
│  kod_stacji  │   km   │ wysokosc_m │
│   varchar    │ double │   double   │
├──────────────┼────────┼────────────┤
│ MzLegZegrzyn │   8.35 │       90.0 │
│ SlZywieKoper │   8.02 │      369.0 │
│ MzPiasPulask │   7.91 │      105.0 │
│ ZpSzczAndrze │   7.43 │       14.0 │
│ MzWarBajkowa │   7.39 │       94.0 │
└──────────────┴────────┴────────────┘



**Różnica wysokości.** Metadane GIOŚ nie zawierają wysokości stacji (arkusz STACJE ma tylko współrzędne), więc różnicy między wysokością stacji a wysokością przyjętą przez model nie da się policzyć z naszych danych. Sprawdzamy w zamian, czy kilka stacji nie dostało pogody z tej samej komórki siatki:

In [57]:
con.sql('''
SELECT
    siatka_szerokosc,
    siatka_dlugosc,
    list(DISTINCT kod_stacji ORDER BY kod_stacji) AS stacje,
    list(DISTINCT wysokosc_m ORDER BY wysokosc_m) AS wysokosci_m
FROM openmeteo_2024
GROUP BY siatka_szerokosc, siatka_dlugosc
HAVING count(DISTINCT kod_stacji) > 1
ORDER BY siatka_szerokosc
''').show(max_rows=40, max_width=250)

┌──────────────────┬────────────────┬────────────────────────────────────────────┬───────────────────────┐
│ siatka_szerokosc │ siatka_dlugosc │                   stacje                   │      wysokosci_m      │
│      double      │     double     │                 varchar[]                  │       double[]        │
├──────────────────┼────────────────┼────────────────────────────────────────────┼───────────────────────┤
│        50.017574 │      21.989529 │ [PkRzeszPilsu, PkRzeszSloci, PkRzeszStarz] │ [206.0, 216.0, 222.0] │
│        50.298767 │      18.980669 │ [SlKatoKossut, SlKatoPlebA4]               │ [274.0, 297.0]        │
│        51.142353 │      16.965889 │ [DsWrocAlWisn, DsWrocWybCon]               │ [116.0, 127.0]        │
│        52.126537 │       21.21547 │ [MzOtwoBrzozo, MzWarBajkowa]               │ [94.0, 101.0]         │
│         52.26713 │      20.961182 │ [MzWarAlNiepo, MzWarTolstoj]               │ [99.0, 119.0]         │
│        52.337433 │           21.0 │

Czy stacje z tej samej komórki mają taką samą pogodę? Porównujemy je godzina po godzinie:

In [58]:
con.sql('''
SELECT
    a.kod_stacji AS stacja_1,
    b.kod_stacji AS stacja_2,
    count(*) AS godziny,
    count(*) FILTER (WHERE a.wind_speed_10m = b.wind_speed_10m
                       AND a.wind_direction_10m = b.wind_direction_10m) AS ten_sam_wiatr,
    count(*) FILTER (WHERE a.precipitation = b.precipitation) AS ten_sam_opad,
    count(*) FILTER (WHERE a.temperature_2m = b.temperature_2m) AS ta_sama_temperatura,
    count(*) FILTER (WHERE a.relative_humidity_2m = b.relative_humidity_2m) AS ta_sama_wilgotnosc,
    count(*) FILTER (WHERE a.surface_pressure = b.surface_pressure) AS to_samo_cisnienie
FROM openmeteo_2024 AS a
JOIN openmeteo_2024 AS b
    ON a.siatka_szerokosc = b.siatka_szerokosc
    AND a.siatka_dlugosc = b.siatka_dlugosc
    AND a.data_czas_utc = b.data_czas_utc
    AND a.kod_stacji < b.kod_stacji
GROUP BY stacja_1, stacja_2
ORDER BY stacja_1, stacja_2
''').show(max_rows=40, max_width=250)

┌──────────────┬──────────────┬─────────┬───────────────┬──────────────┬─────────────────────┬────────────────────┬───────────────────┐
│   stacja_1   │   stacja_2   │ godziny │ ten_sam_wiatr │ ten_sam_opad │ ta_sama_temperatura │ ta_sama_wilgotnosc │ to_samo_cisnienie │
│   varchar    │   varchar    │  int64  │     int64     │    int64     │        int64        │       int64        │       int64       │
├──────────────┼──────────────┼─────────┼───────────────┼──────────────┼─────────────────────┼────────────────────┼───────────────────┤
│ DsWrocAlWisn │ DsWrocWybCon │    8784 │          8784 │         8784 │                8784 │               8784 │                 0 │
│ KpBydPlPozna │ KpBydWarszaw │    8784 │          8784 │         8784 │                8784 │               8784 │                 0 │
│ MzLegZegrzyn │ MzWarKondrat │    8784 │          8784 │         8784 │                8784 │               8784 │                 0 │
│ MzOtwoBrzozo │ MzWarBajkowa │    8784 │       

9 komórek siatki obsługuje po 2-3 stacje, razem 19 z 96 stacji (stacje położone blisko siebie: we Wrocławiu, Rzeszowie, Katowicach, Bydgoszczy, Białymstoku, Gdańsku oraz w Warszawie i okolicach). Stacje z jednej komórki mają przez cały rok identyczną temperaturę, wilgotność, wiatr i opad - w każdej z 8784 godzin. Różni się tylko ciśnienie, przeliczone dla innej wysokości terenu (`wysokosc_m` różni się w obrębie komórki, więc opisuje punkt stacji, a nie komórkę).

Dla modelu oznacza to, że pogoda nie odróżnia tych stacji od siebie - różnice w PM2.5 między nimi muszą wynikać z innych cech.

## 6. Porównanie źródeł (pomiary GIOŚ i pogoda)

W warstwie silver (Milestone 2) PM2.5 zostanie złączone z pogodą po stacji i godzinie. Przed złączeniem sprawdzamy, czy klucze się pokrywają i czy oba źródła liczą czas tak samo.

### 6.1. Pokrycie stacji

In [59]:
con.sql('''
WITH pm25 AS (
    SELECT DISTINCT kod_stacji
    FROM pomiary_2024
    WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
),
pogoda AS (
    SELECT DISTINCT kod_stacji FROM openmeteo_2024
)
SELECT
    (SELECT count(*) FROM pm25) AS stacje_pm25,
    (SELECT count(*) FROM pogoda) AS stacje_z_pogoda,
    (SELECT count(*) FROM pm25 ANTI JOIN pogoda USING (kod_stacji)) AS pm25_bez_pogody,
    (SELECT count(*) FROM pogoda ANTI JOIN pm25 USING (kod_stacji)) AS pogoda_bez_pm25,
    (SELECT count(*) FROM pm25 ANTI JOIN stacje USING (kod_stacji)) AS pm25_bez_wpisu_w_stacje
''').show(max_rows=40, max_width=250)

┌─────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────────────┐
│ stacje_pm25 │ stacje_z_pogoda │ pm25_bez_pogody │ pogoda_bez_pm25 │ pm25_bez_wpisu_w_stacje │
│    int64    │      int64      │      int64      │      int64      │          int64          │
├─────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────────────┤
│          96 │              96 │               0 │               0 │                       0 │
└─────────────┴─────────────────┴─────────────────┴─────────────────┴─────────────────────────┘



Pokrycie jest pełne z konstrukcji: pogodę pobraliśmy dla kodów stacji z nagłówka pliku `2024_PM25_1g.xlsx`, a współrzędne wzięliśmy z metadanych. Dwa stanowiska PM2.5 (`SlZywieKoper`, `ZpKolZolkiew`) nie mają wpisu w słowniku stanowisk (4.5), ale ich stacje są w słowniku stacji, więc złączenie z pogodą po `kod_stacji` je obejmie.

### 6.2. Strefa czasowa GIOŚ

Open-Meteo podaje czas w UTC. GIOŚ zapisuje czas bez strefy, a czas urzędowy w Polsce zmienia się dwa razy w roku (UTC+1 zimą, UTC+2 latem). Gdyby GIOŚ używał czasu urzędowego, 31 marca brakowałoby godziny 02:00-03:00, a 27 października jedna godzina powtórzyłaby się dwa razy.

**Test 1: zmiana czasu.** Liczba rekordów godzinowych w okolicy obu zmian czasu:

In [60]:
con.sql('''
SELECT data_czas, count(*) AS rekordy
FROM pomiary_bez_dubli
WHERE czas_usredniania = '1g'
    AND (
        data_czas BETWEEN TIMESTAMP '2024-03-31 01:00' AND TIMESTAMP '2024-03-31 04:00'
        OR data_czas BETWEEN TIMESTAMP '2024-10-27 01:00' AND TIMESTAMP '2024-10-27 04:00'
    )
GROUP BY data_czas
ORDER BY data_czas
''').show(max_rows=40, max_width=250)

┌─────────────────────┬─────────┐
│      data_czas      │ rekordy │
│      timestamp      │  int64  │
├─────────────────────┼─────────┤
│ 2024-03-31 01:00:00 │     986 │
│ 2024-03-31 02:00:00 │     986 │
│ 2024-03-31 03:00:00 │     986 │
│ 2024-03-31 04:00:00 │     986 │
│ 2024-10-27 01:00:00 │     986 │
│ 2024-10-27 02:00:00 │     986 │
│ 2024-10-27 03:00:00 │     986 │
│ 2024-10-27 04:00:00 │     986 │
└─────────────────────┴─────────┘



Wszystkie godziny występują po jednym razie dla każdego z 986 stanowisk, a cały rok ma dokładnie 8784 godziny (3.6). GIOŚ nie stosuje więc czasu letniego - przesunięcie względem UTC jest stałe przez cały rok.

**Test 2: jakie przesunięcie.** Ozon powstaje w słońcu, więc rośnie razem z temperaturą, a PM2.5 jest najwyższe w zimne godziny. Przesuwamy pogodę o `k` godzin względem znacznika GIOŚ (`data_czas_utc = data_czas + k h`) i sprawdzamy, przy jakim `k` korelacja jest najsilniejsza:

In [61]:
con.sql('''
WITH pomiar AS (
    SELECT kod_stacji, data_czas, wartosc
    FROM pomiary_bez_dubli
    WHERE wskaznik = 'O3' AND czas_usredniania = '1g' AND wartosc IS NOT NULL
)
SELECT
    k AS przesuniecie_h,
    round(corr(p.wartosc, w.temperature_2m), 4) AS korelacja_z_temperatura,
    count(*) AS pary
FROM pomiar AS p
CROSS JOIN range(-4, 3) AS t(k)
JOIN openmeteo_2024 AS w
    ON w.kod_stacji = p.kod_stacji
    AND w.data_czas_utc = p.data_czas + to_hours(k)
GROUP BY k
ORDER BY k
''').show(max_rows=40, max_width=250)

┌────────────────┬─────────────────────────┬────────┐
│ przesuniecie_h │ korelacja_z_temperatura │  pary  │
│     int64      │         double          │ int64  │
├────────────────┼─────────────────────────┼────────┤
│             -4 │                   0.562 │ 345138 │
│             -3 │                  0.5849 │ 345177 │
│             -2 │                  0.5935 │ 345216 │
│             -1 │                  0.5856 │ 345255 │
│              0 │                  0.5634 │ 345215 │
│              1 │                  0.5312 │ 345175 │
│              2 │                  0.4928 │ 345136 │
└────────────────┴─────────────────────────┴────────┘



In [62]:
con.sql('''
WITH pomiar AS (
    SELECT kod_stacji, data_czas, wartosc
    FROM pomiary_bez_dubli
    WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g' AND wartosc IS NOT NULL
)
SELECT
    k AS przesuniecie_h,
    round(corr(p.wartosc, w.temperature_2m), 4) AS korelacja_z_temperatura,
    count(*) AS pary
FROM pomiar AS p
CROSS JOIN range(-4, 3) AS t(k)
JOIN openmeteo_2024 AS w
    ON w.kod_stacji = p.kod_stacji
    AND w.data_czas_utc = p.data_czas + to_hours(k)
GROUP BY k
ORDER BY k
''').show(max_rows=40, max_width=250)

┌────────────────┬─────────────────────────┬────────┐
│ przesuniecie_h │ korelacja_z_temperatura │  pary  │
│     int64      │         double          │ int64  │
├────────────────┼─────────────────────────┼────────┤
│             -4 │                 -0.3266 │ 817384 │
│             -3 │                 -0.3345 │ 817473 │
│             -2 │                 -0.3377 │ 817562 │
│             -1 │                 -0.3361 │ 817648 │
│              0 │                 -0.3308 │ 817564 │
│              1 │                 -0.3234 │ 817481 │
│              2 │                 -0.3148 │ 817398 │
└────────────────┴─────────────────────────┴────────┘



Oba wskaźniki mają najsilniejszą korelację przy `k = -2` - wyraźnie dla ozonu (0,594 wobec 0,586 przy −1 i −3), słabiej dla PM2.5, gdzie sąsiednie przesunięcia różnią się o tysięczne części: pomiar GIOŚ najlepiej pasuje do pogody z godziny UTC zaczynającej się 2 godziny przed znacznikiem. To zgadza się z hipotezą, że GIOŚ zapisuje **koniec** godziny (−1 h) w czasie **UTC+1** (−1 h). Przykład: GIOŚ `2024-07-01 14:00` to średnia z 13:00-14:00 czasu UTC+1, czyli 12:00-13:00 UTC, czyli wiersz Open-Meteo `2024-07-01 12:00`.

To test pośredni: między temperaturą a ozonem jest też fizyczne opóźnienie, więc wynik wspiera hipotezę, ale jej nie dowodzi. Konwencję trzeba potwierdzić w dokumentacji GIOŚ przed złączeniem w Milestone 2. Dodatkowo w Open-Meteo opad to suma z poprzedniej godziny (wiersz `13:00` = opad 12:00-13:00), więc dla opadu pasujący wiersz jest o godzinę późniejszy niż dla temperatury.

### 6.3. Pokrycie czasu po dopasowaniu

Ile godzinowych rekordów PM2.5 znajdzie parę w pogodzie przy przesunięciu −2 h?

In [63]:
con.sql('''
SELECT
    count(*) AS rekordy_pm25,
    count(w.kod_stacji) AS z_pogoda,
    count(*) - count(w.kod_stacji) AS bez_pogody,
    min(p.data_czas) FILTER (WHERE w.kod_stacji IS NULL) AS bez_pogody_od,
    max(p.data_czas) FILTER (WHERE w.kod_stacji IS NULL) AS bez_pogody_do
FROM pomiary_bez_dubli AS p
LEFT JOIN openmeteo_2024 AS w
    ON w.kod_stacji = p.kod_stacji
    AND w.data_czas_utc = p.data_czas - INTERVAL 2 HOUR
WHERE p.wskaznik = 'PM2.5' AND p.czas_usredniania = '1g'
''').show(max_rows=40, max_width=250)

┌──────────────┬──────────┬────────────┬─────────────────────┬─────────────────────┐
│ rekordy_pm25 │ z_pogoda │ bez_pogody │    bez_pogody_od    │    bez_pogody_do    │
│    int64     │  int64   │   int64    │      timestamp      │      timestamp      │
├──────────────┼──────────┼────────────┼─────────────────────┼─────────────────────┤
│       843264 │   843168 │         96 │ 2024-01-01 01:00:00 │ 2024-01-01 01:00:00 │
└──────────────┴──────────┴────────────┴─────────────────────┴─────────────────────┘



Bez pary zostaje tylko pierwsza godzina roku na każdej stacji (`2024-01-01 01:00` GIOŚ = `2023-12-31 23:00` UTC), bo pogodę pobraliśmy od `2024-01-01 00:00` UTC. Po przesunięciu nie ma też pary ostatnia godzina pogody (`2024-12-31 23:00` UTC).

## 7. Tabela zbiorcza

Jedno zestawienie wykrytych anomalii z liczbą przypadków i udziałem w całości. Podstawa procentu (`baza`) zależy od anomalii: np. wartości ujemne liczymy względem niepustych pomiarów, a brakujące wpisy w słowniku - względem stanowisk, które mają pomiary. Wszystkie liczby pochodzą z zapytań z sekcji 3-6.

In [64]:
con.sql('''
WITH
pomiary AS (
    SELECT
        count(*) AS rekordy,
        count(wartosc) AS niepuste,
        count(*) - count(wartosc) AS braki,
        count(*) - count(DISTINCT (kod_stanowiska, data_czas)) AS duble,
        count(*) FILTER (WHERE wartosc < 0) AS ujemne,
        count(*) FILTER (WHERE wartosc = 0) AS zera
    FROM pomiary_2024
),
pyl AS (
    SELECT
        count(*) FILTER (WHERE pm25 IS NOT NULL AND pm10 IS NOT NULL) AS baza,
        count(*) FILTER (WHERE pm25 > pm10) AS n
    FROM (
        SELECT
            kod_stacji,
            data_czas,
            max(wartosc) FILTER (WHERE wskaznik = 'PM2.5') AS pm25,
            max(wartosc) FILTER (WHERE wskaznik = 'PM10') AS pm10
        FROM pomiary_2024
        WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
        GROUP BY kod_stacji, data_czas
    )
),
tlenki AS (
    SELECT
        count(*) FILTER (WHERE no2 IS NOT NULL AND nox IS NOT NULL) AS baza,
        count(*) FILTER (WHERE no2 > nox) AS n
    FROM (
        SELECT
            kod_stacji,
            data_czas,
            max(wartosc) FILTER (WHERE wskaznik = 'NO2') AS no2,
            max(wartosc) FILTER (WHERE wskaznik = 'NOx') AS nox
        FROM pomiary_2024
        WHERE czas_usredniania = '1g' AND wskaznik IN ('NO2', 'NOx')
        GROUP BY kod_stacji, data_czas
    )
),
piki AS (
    SELECT
        count(wartosc) AS baza,
        count(*) FILTER (WHERE wzrost > 50 AND spadek > 50) AS n
    FROM (
        SELECT
            wartosc,
            wartosc - lag(wartosc) OVER w AS wzrost,
            wartosc - lead(wartosc) OVER w AS spadek
        FROM pomiary_bez_dubli
        WHERE czas_usredniania = '1g' AND wskaznik = 'PM2.5'
        WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
    )
),
stale AS (
    SELECT
        sum(godziny) AS baza,
        sum(godziny) FILTER (WHERE wartosc IS NOT NULL AND godziny >= 12) AS n
    FROM ciagi
    WHERE wskaznik = 'PM2.5'
),
luki AS (
    SELECT
        sum(godziny) AS baza,
        sum(godziny) FILTER (WHERE wartosc IS NULL AND godziny >= 24) AS n
    FROM ciagi
    WHERE wskaznik = 'PM2.5'
),
kompletnosc AS (
    SELECT count(*) AS baza, count(*) FILTER (WHERE proc < 90) AS n
    FROM (
        SELECT kod_stanowiska, 100 * count(wartosc) / count(*) AS proc
        FROM pomiary_2024
        WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
        GROUP BY kod_stanowiska
    )
),
st AS (
    SELECT
        count(*) AS baza,
        count(*) FILTER (WHERE szerokosc = -999 OR dlugosc = -999) AS minus_999,
        count(*) FILTER (WHERE typ_stacji = 'BRAK DANYCH' OR rodzaj_stacji = 'BRAK DANYCH') AS brak_danych,
        count(*) - count(DISTINCT kod_stacji) AS duble
    FROM stacje
),
sn AS (
    SELECT count(*) AS baza, count(*) - count(DISTINCT kod_stanowiska) AS duble
    FROM stanowiska
),
klucze AS (
    SELECT
        (SELECT count(DISTINCT kod_stanowiska) FROM pomiary_2024) AS stanowiska_baza,
        (SELECT count(*) FROM (SELECT DISTINCT kod_stanowiska FROM pomiary_2024)
            ANTI JOIN stanowiska USING (kod_stanowiska)) AS stanowiska_bez,
        (SELECT count(DISTINCT kod_stacji) FROM pomiary_2024) AS stacje_baza,
        (SELECT count(*) FROM (SELECT DISTINCT kod_stacji FROM pomiary_2024)
            ANTI JOIN stacje USING (kod_stacji)) AS stacje_bez
),
okres AS (
    SELECT
        count(*) AS baza,
        count(*) FILTER (WHERE pierwszy < uruchomienie OR ostatni > zamkniecie) AS n
    FROM (
        SELECT
            kod_stanowiska,
            min(CASE WHEN czas_usredniania = '1g' THEN data_czas - INTERVAL 1 HOUR ELSE data_czas END)::DATE AS pierwszy,
            max(CASE WHEN czas_usredniania = '1g' THEN data_czas - INTERVAL 1 HOUR ELSE data_czas END)::DATE AS ostatni
        FROM pomiary_2024
        WHERE wartosc IS NOT NULL
        GROUP BY kod_stanowiska
    )
    JOIN (
        SELECT
            kod_stanowiska,
            any_value(data_uruchomienia) AS uruchomienie,
            any_value(data_zamkniecia) AS zamkniecie
        FROM stanowiska
        GROUP BY kod_stanowiska
    ) USING (kod_stanowiska)
),
pogoda AS (
    SELECT
        count(*) AS baza,
        count(*) FILTER (
            WHERE relative_humidity_2m NOT BETWEEN 0 AND 100
            OR precipitation < 0
            OR wind_speed_10m < 0
            OR wind_direction_10m NOT BETWEEN 0 AND 360
        ) AS niemozliwe,
        count(*) FILTER (WHERE wind_speed_10m = 0) AS cisza
    FROM openmeteo_2024
),
siatka AS (
    SELECT sum(stacje) AS baza, sum(stacje) FILTER (WHERE stacje > 1) AS n
    FROM (
        SELECT count(DISTINCT kod_stacji) AS stacje
        FROM openmeteo_2024
        GROUP BY siatka_szerokosc, siatka_dlugosc
    )
),
zestawienie AS (
    SELECT 1 AS nr, 'pomiary' AS obszar, 'brak pomiaru (NULL w wartosc)' AS anomalia, braki AS liczba, 'rekordy pomiarów' AS baza_opis, rekordy AS baza FROM pomiary
    UNION ALL SELECT 2, 'pomiary', 'zdublowany klucz (stanowisko, czas)', duble, 'rekordy pomiarów', rekordy FROM pomiary
    UNION ALL SELECT 3, 'pomiary', 'wartość ujemna', ujemne, 'niepuste pomiary', niepuste FROM pomiary
    UNION ALL SELECT 4, 'pomiary', 'wartość dokładnie 0', zera, 'niepuste pomiary', niepuste FROM pomiary
    UNION ALL SELECT 5, 'pomiary', 'PM2.5 większe od PM10 (ta sama stacja i godzina)', n, 'godziny z PM2.5 i PM10', baza FROM pyl
    UNION ALL SELECT 6, 'pomiary', 'NO2 większe od NOx (ta sama stacja i godzina)', n, 'godziny z NO2 i NOx', baza FROM tlenki
    UNION ALL SELECT 7, 'pomiary', 'jednogodzinny pik PM2.5 (> 50 ug/m3 ponad obie sąsiednie godziny)', n, 'pomiary PM2.5 1g', baza FROM piki
    UNION ALL SELECT 8, 'pomiary', 'PM2.5 stałe przez >= 12 h (godziny w takich ciągach)', n, 'godziny PM2.5 1g', baza FROM stale
    UNION ALL SELECT 9, 'pomiary', 'luka PM2.5 >= 24 h (godziny w takich lukach)', n, 'godziny PM2.5 1g', baza FROM luki
    UNION ALL SELECT 10, 'pomiary', 'stanowisko PM2.5 z kompletnością < 90%', n, 'stanowiska PM2.5 1g', baza FROM kompletnosc
    UNION ALL SELECT 11, 'metadane', 'współrzędne -999 zamiast braku', minus_999, 'wiersze stacje', baza FROM st
    UNION ALL SELECT 12, 'metadane', 'BRAK DANYCH w typie albo rodzaju stacji', brak_danych, 'wiersze stacje', baza FROM st
    UNION ALL SELECT 13, 'metadane', 'zdublowany kod stacji (nadmiarowe wiersze)', duble, 'wiersze stacje', baza FROM st
    UNION ALL SELECT 14, 'metadane', 'zdublowany kod stanowiska (nadmiarowe wiersze)', duble, 'wiersze stanowiska', baza FROM sn
    UNION ALL SELECT 15, 'metadane', 'stanowisko z pomiarów bez wpisu w słowniku', stanowiska_bez, 'stanowiska z pomiarami', stanowiska_baza FROM klucze
    UNION ALL SELECT 16, 'metadane', 'stacja z pomiarów pod innym kodem niż w słowniku', stacje_bez, 'stacje z pomiarami', stacje_baza FROM klucze
    UNION ALL SELECT 17, 'metadane', 'pomiary poza okresem działania stanowiska', n, 'stanowiska ze słownika z pomiarami', baza FROM okres
    UNION ALL SELECT 18, 'pogoda', 'wartość fizycznie niemożliwa', niemozliwe, 'rekordy pogody', baza FROM pogoda
    UNION ALL SELECT 19, 'pogoda', 'cisza z kierunkiem wiatru 270° (wartość stała)', cisza, 'rekordy pogody', baza FROM pogoda
    UNION ALL SELECT 20, 'pogoda', 'stacja dzieląca komórkę siatki z inną (identyczna pogoda)', n, 'stacje z pogodą', baza FROM siatka
)
SELECT
    nr,
    obszar,
    anomalia,
    liczba::BIGINT AS liczba,
    baza_opis,
    baza::BIGINT AS baza,
    round(100 * liczba / baza, 3) AS procent
FROM zestawienie
ORDER BY nr
''').show(max_rows=40, max_width=250)

┌───────┬──────────┬───────────────────────────────────────────────────────────────────┬────────┬────────────────────────────────────┬─────────┬─────────┐
│  nr   │  obszar  │                             anomalia                              │ liczba │             baza_opis              │  baza   │ procent │
│ int32 │ varchar  │                              varchar                              │ int64  │              varchar               │  int64  │ double  │
├───────┼──────────┼───────────────────────────────────────────────────────────────────┼────────┼────────────────────────────────────┼─────────┼─────────┤
│     1 │ pomiary  │ brak pomiaru (NULL w wartosc)                                     │ 470793 │ rekordy pomiarów                   │ 8952726 │   5.259 │
│     2 │ pomiary  │ zdublowany klucz (stanowisko, czas)                               │   8784 │ rekordy pomiarów                   │ 8952726 │   0.098 │
│     3 │ pomiary  │ wartość ujemna                                   

W tabeli nie ma anomalii, których nie da się policzyć jako liczby rekordów:

- przesunięcie czasu GIOŚ względem UTC (−2 h według testu z 6.2),
- 4 daty zamknięcia stanowisk `31.12.20025`, które w bronze są już `NULL` (wykryte w notebooku 02),
- skrajne wartości SO2 i PM z 3.4, które układają się w epizody, a nie w pojedyncze błędy.

## 8. Anomalie techniczne

Opis anomalii wykrytych w profilowaniu: co widać w danych, jak często występuje i gdzie to sprawdzić. Pole **Możliwe źródło** opisuje prawdopodobną przyczynę; źródła informacji są na końcu sekcji 8. Tam, gdzie dokumentacja nic nie mówi, przyczyna jest oznaczona jako hipoteza.

Liczby pochodzą z wyników zapytań w sekcjach 3-7.

### 8.1. Braki pomiarów

**Co widać:** okresy bez pomiaru zapisane jako `NULL` w `wartosc`. Siatka czasu jest pełna, więc brak pomiaru to pusta wartość, a nie brakujący wiersz.

**Skala:** 470 793 rekordy (5,26% wszystkich). Główne wskaźniki mają kompletność 95-98%, rtęć 83,3%, formaldehyd 42,6%, a lotne związki organiczne z Borów Tucholskich od 2,7% do 63,4%. Dla PM2.5: 3 z 96 stanowisk poniżej 90%, 127 luk trwających co najmniej dobę (20 176 godzin, 2,39%), najsłabszy miesiąc to grudzień (93,3%).

**Przykład:** `MzSiedKonars-PM2.5-1g` nie ma pomiarów od `2024-06-07 00:00` do końca roku (4993 godziny, kompletność 42,7%); w metadanych stanowisko ma datę zamknięcia 2024-06-06.

**Gdzie w notebooku:** 3.1, 3.2, 3.6

**Możliwe źródło:** Błąd techniczny: awarie, serwis i kalibracje przyrządów, przerwy w transmisji oraz wartości odrzucone przy weryfikacji. Część braków wynika z zamknięcia stanowiska w trakcie roku (np. `MzSiedKonars` zamknięte 2024-06-06). Niska kompletność formaldehydu i lotnych związków organicznych wynika z pomiarów prowadzonych tylko w części roku.

### 8.2. To samo stanowisko w dwóch plikach

**Co widać:** jedno stanowisko występuje w dwóch plikach źródłowych, więc każda jego godzina jest w bronze dwa razy.

**Skala:** 8784 nadmiarowe wiersze (0,098% rekordów), jedno stanowisko; wartości w obu kopiach są identyczne we wszystkich 8784 godzinach.

**Przykład:** `KpZielBoryTu-C6H6-1g` jest w `2024_C6H6_1g.xlsx` i w `2024_PrekursoryZielonka_1g.xlsx`.

**Gdzie w notebooku:** 3.5

**Możliwe źródło:** Cecha formatu danych: benzen z Borów Tucholskich jest mierzony w programie prekursorów ozonu i trafia zarówno do pliku tego programu, jak i do pliku benzenu, bo benzen jest też wskaźnikiem regulowanym. To ten sam pomiar wyeksportowany dwa razy, a nie dwa przyrządy.

### 8.3. Wartości ujemne

**Co widać:** stężenie mniejsze od zera, czyli wartość niemożliwa fizycznie.

**Skala:** 2 pomiary na 8 481 933 niepustych, jeden wskaźnik.

**Przykład:** `WmPuszczaBor-OC(PM2.5)-24g` (węgiel organiczny w PM2.5): −0,35 µg/m³ 16.12.2024 i −0,37 µg/m³ 17.12.2024.

**Gdzie w notebooku:** 3.4

**Możliwe źródło:** Błąd techniczny typowy dla metody: OC mierzy się z filtra metodą termo-optyczną, od której odejmuje się wynik filtra ślepego. Przy bardzo niskich stężeniach (czyste powietrze w Puszczy Boreckiej) wynik po odjęciu może wyjść lekko poniżej zera, a rzeczywiste stężenie jest bliskie zeru. Hipoteza - dokumentacja GIOŚ tego nie opisuje.

### 8.4. Wartości dokładnie zero

**Co widać:** pomiary równe dokładnie `0.0`. Dla pyłu zero oznacza całkowity brak cząstek w powietrzu.

**Skala:** 26 007 pomiarów (0,31% niepustych), najwięcej w NO (14 856, 1,31%), SO2 (6 489) i benzenie (2 382). W pyle 728 zer PM2.5 i 990 zer PM10 - skupionych na kilku stanowiskach.

**Przykład:** `PdBorsukowiz`: 427 zer PM2.5 i 306 zer PM10; kolejne `PdSuwPulask2-PM2.5-1g` (198) i `DsDziePilsud-PM10-1g` (185).

**Gdzie w notebooku:** 3.4

**Możliwe źródło:** Dla gazów (NO, SO2, benzen) głównie cecha pomiaru: stężenia poniżej progu wykrywalności analizatora zapisywane są jako 0. Dla pyłu 0 µg/m³ przez wiele godzin jest fizycznie nierealne - najpewniej błąd przyrządu albo przycinanie ujemnych odczytów do zera przy bardzo niskich stężeniach na stacji tła (`PdBorsukowiz`). Hipoteza - dokumentacja GIOŚ nie opisuje traktowania wartości poniżej progu wykrywalności.

### 8.5. Stała wartość przez wiele godzin (zawieszony czujnik)

**Co widać:** ta sama liczba zapisana w wielu kolejnych godzinach, choć stężenia pyłu zwykle zmieniają się z godziny na godzinę (mediana zmiany PM2.5 to 1,0 µg/m³).

**Skala:** PM2.5: 15 ciągów od 12 godzin (270 godzin, 0,032%), w tym 10 ciągów zer; najdłuższy ciąg niezerowy 17 godzin. Gazy o niskich stężeniach mają wiele długich ciągów: ciągi od doby to 64 dla SO2, 50 dla benzenu i 39 dla NO.

**Przykład:** `PdBorsukowiz`: PM2.5 i PM10 jednocześnie równe 0,0 przez 30 godzin, od `2024-02-17 18:00` do `2024-02-18 23:00`. Niezerowy: `PmBytowMila1MOB-PM2.5-1g` = 1,0 przez 17 godzin (16-17.12.2024).

**Gdzie w notebooku:** 3.6, 3.7

**Możliwe źródło:** Błąd techniczny: zawieszony czujnik albo system, który przy braku nowych danych powtarza ostatni odczyt. Dla gazów o niskich stężeniach także cecha zapisu - przy zaokrągleniu do 0,1-1 µg/m³ kolejne godziny naturalnie dają tę samą liczbę.

### 8.6. Jednogodzinne piki

**Co widać:** wartość wyższa o ponad 50 µg/m³ zarówno od poprzedniej, jak i od następnej godziny - odczyt wyskakuje na jedną godzinę i od razu wraca.

**Skala:** PM2.5: 91 pików ponad 50 µg/m³ i 14 ponad 100 µg/m³ (0,011% pomiarów); PM10: 530 i 83. Piki PM2.5 powtarzają się na tych samych stanowiskach: 19 z 91 na `WmDzialPolnaMOB`.

**Przykład:** `WmDzialPolnaMOB-PM2.5-1g`, `2024-03-09 18:00`: 42,7 → 365,0 → 67,0 µg/m³.

**Gdzie w notebooku:** 3.8

**Możliwe źródło:** Częściowo zjawisko rzeczywiste: krótkie lokalne źródło przy stacji (dym z komina, ognisko, pojazd, prace). Częściowo błąd techniczny, np. zakłócenie po restarcie lub kalibracji przyrządu. Koncentracja na stacjach mobilnych sugeruje wpływ ich tymczasowej lokalizacji, często bliżej źródeł emisji.

### 8.7. Niespójność między wskaźnikami tej samej stacji

**Co widać:** PM2.5 większe od PM10 albo NO2 większe od NOx w tej samej godzinie, choć z definicji PM2.5 ≤ PM10 i NO2 ≤ NOx.

**Skala:** PM2.5 > PM10: 1003 godziny na 8 stacjach (0,13% godzin z oboma pomiarami), najwyżej o 6,4 µg/m³. NO2 > NOx: 5220 godzin na 34 stacjach (0,54%), w tym 10 razy o ponad 5 µg/m³, najwyżej o 9,9 µg/m³.

**Przykład:** różnice są małe w porównaniu z typowymi poziomami (mediana PM2.5 11,5, NO2 9,8 µg/m³).

**Gdzie w notebooku:** 3.4

**Możliwe źródło:** Błąd techniczny w granicach niepewności pomiaru: PM2.5 i PM10 mierzą dwa osobne przyrządy, a NO2 wylicza się jako różnicę NOx i NO z tego samego analizatora. Przy niskich stężeniach różnice kalibracji i zaokrąglenia dają małe odwrócenia, co widać po skali kilku µg/m³.

### 8.8. Wartości skrajne

**Co widać:** pojedyncze wartości daleko powyżej reszty rozkładu. Do oceny, czy to błąd przyrządu, czy rzeczywisty epizod - skrajne wartości pojawiają się w tych samych godzinach na kilku stacjach.

**Skala:** SO2: maksimum 1353,8 µg/m³, 28 razy więcej niż p99,9 (47,6). PM2.5: maksimum 450,5 µg/m³ (3 razy p99,9). NOx: 1104,5 µg/m³.

**Przykład:** SO2 27.02.2024 o 13:00 na dwóch stacjach w Szczecinie (`ZpSzczLaczna` 1353,8 i `ZpSzczPilsud` 814,5). PM2.5 i PM10 osiągają maksima na kilku stacjach w dniach 8-10.01.2024 (np. `PdAugustowUz` 450,5 µg/m³ 08.01 o 22:00).

**Gdzie w notebooku:** 3.3, 3.4

**Możliwe źródło:** Zjawisko rzeczywiste, potwierdzone komunikatami GIOŚ. 8-10.01.2024: ogólnopolski epizod smogowy (mróz, inwersja, słaby wiatr, ogrzewanie paliwami stałymi); komunikat z 09.01 wymienia ryzyko przekroczenia poziomu alarmowego m.in. w powiecie augustowskim. 27.02.2024: epizod SO2 w Szczecinie z powiadomieniem o ryzyku przekroczenia poziomu alarmowego, najpewniej z emisji przemysłowej (źródła nie ustalono). Tych wartości nie należy usuwać przy czyszczeniu.

### 8.9. Konwencja czasu GIOŚ

**Co widać:** znacznik czasu GIOŚ nie ma strefy i oznacza koniec okresu uśredniania: pomiary godzinowe zaczynają się o `2024-01-01 01:00` i kończą o `2025-01-01 00:00`. GIOŚ nie stosuje czasu letniego. Test korelacji z temperaturą wskazuje przesunięcie −2 h względem UTC (koniec godziny w UTC+1) - to hipoteza, niepotwierdzona w dokumentacji.

**Skala:** 987 rekordów z datą w 2025 r. (jeden na stanowisko godzinowe plus kopia z 8.2); po przesunięciu o −2 h 96 rekordów PM2.5 (pierwsza godzina roku na każdej stacji) nie ma pary w pogodzie.

**Przykład:** GIOŚ `2024-07-01 14:00` = średnia 12:00-13:00 UTC = Open-Meteo `2024-07-01 12:00` (przy przyjętej hipotezie).

**Gdzie w notebooku:** 3.6, 6.2, 6.3

**Możliwe źródło:** Cecha formatu danych. Strefa potwierdzona w dokumentacji API GIOŚ: dane archiwalne są w czasie UTC+01:00 (środkowoeuropejski zimowy), bez czasu letniego. To, że godzina oznacza koniec okresu uśredniania, wynika z zakresu danych (01:00 do 00:00 następnego roku) i testu z 6.2 - dokumentacja GIOŚ tego nie opisuje.

### 8.10. Kody w pomiarach niezgodne ze słownikiem

**Co widać:** część kodów stacji i stanowisk z plików pomiarów nie występuje w metadanych, więc złączenie po kodzie je gubi.

**Skala:** 17 z 1759 stanowisk z pomiarami nie ma wpisu w słowniku stanowisk (w tym 8 stanowisk NO na stacjach, które mają w słowniku NO2 i NOx). 2 z 286 stacji z pomiarami mają w słowniku inny kod. 6 wierszy słownika stanowisk wskazuje stacje, których nie ma w słowniku stacji.

**Przykład:** `ZpSzczLaczna` w pomiarach to `ZpSzczLacz04` w metadanych (Szczecin, ul. Łączna); `OpBrzegPoprzMOB` to `OpBrzegPoprz` (Brzeg, ul. Poprzeczna).

**Gdzie w notebooku:** 4.5

**Możliwe źródło:** Cecha formatu danych: stacje zmieniają kody (np. po zmianie lokalizacji albo przejściu ze stacji mobilnej na stałą), a plik metadanych jest aktualny na dzień pobrania, podczas gdy pomiary zawierają kody z 2024 r.

### 8.11. Wartości zastępcze w metadanych

**Co widać:** zamiast pustego pola wpisany tekst albo liczba spoza zakresu, które nie są `NULL` i nie wychwyci ich zwykłe liczenie braków.

**Skala:** 76 z 1148 stacji ma współrzędne `-999` (żadna nie działała w 2024 r.); `BRAK DANYCH` 25 razy (21× rodzaj stacji, 4× typ stacji); 4 daty zamknięcia stanowisk zapisane jako `31.12.20025`, zamienione w bronze na `NULL`.

**Przykład:** `LdAlekslWSSEMSklodow`: szerokość i długość `-999`.

**Gdzie w notebooku:** 4.1, 4.2, notebook 02 (kontrola spójności)

**Możliwe źródło:** Cecha formatu danych: ręcznie prowadzony słownik, w którym brak informacji wpisywano jako `-999` lub `BRAK DANYCH` (głównie dawno zamknięte stacje). Data `31.12.20025` to literówka przy ręcznym wpisie.

### 8.12. Duplikaty i sprzeczności w metadanych

**Co widać:** zdublowane kody i daty, które sobie przeczą.

**Skala:** 1 zdublowany kod stacji i 6 zdublowanych kodów stanowisk (wiersze identyczne); 1 stacja z datą zamknięcia przed datą uruchomienia; 1 stacja ze starym kodem równym obecnemu.

**Przykład:** `OpKrap3Maja` ma 2 wiersze różniące się tylko starym kodem; `PdSuwPulaski` ma po 2 identyczne wiersze dla 6 stanowisk WWA; `OpNysaRodzi2`: uruchomienie 2025-01-01, zamknięcie 2024-12-31.

**Gdzie w notebooku:** 4.4

**Możliwe źródło:** Błąd zapisu przy ręcznym prowadzeniu słownika: zdublowane wiersze i daty sprzeczne ze sobą (zamknięcie przed uruchomieniem).

### 8.13. Pomiary poza okresem działania stanowiska

**Co widać:** pomiary z dni, w których według metadanych stanowisko jeszcze albo już nie działało.

**Skala:** 9 z 1742 stanowisk, wszystkie na stacjach mobilnych (`MOB`): 5 ma pomiary od 5 do 19 dni po dacie zamknięcia, 4 - pomiar z dnia przed uruchomieniem.

**Przykład:** `MpMuszynLodoMOB-BaP(PM10)-24g`: zamknięte 2024-12-12, ostatni pomiar 2024-12-31.

**Gdzie w notebooku:** 4.6

**Możliwe źródło:** Stacje mobilne prowadzą kampanie pomiarowe w różnych miejscach (GIOŚ kupił je m.in. do monitoringu uzdrowisk), więc daty w metadanych odpowiadają planowi kampanii, a przyrząd mierzył jeszcze kilka dni dłużej. Możliwe też przesunięcie dat w słowniku.

### 8.14. Kierunek wiatru przy ciszy i kodowanie północy

**Co widać:** przy prędkości wiatru 0 kierunek ma zawsze tę samą wartość 270°; północ jest zapisywana jako 360°, a 0° pojawia się raz.

**Skala:** 106 godzin ciszy (0,013% rekordów pogody), wszystkie z kierunkiem 270°; 2854 rekordy z kierunkiem 360° i 1 z kierunkiem 0°.

**Przykład:** każdy rekord z `wind_speed_10m = 0` ma `wind_direction_10m = 270`.

**Gdzie w notebooku:** 5.3

**Możliwe źródło:** Cecha formatu Open-Meteo: kierunek liczy się ze składowych wiatru, a przy prędkości 0 jest nieokreślony i model wpisuje stałą wartość (270°). Północ zapisywana jest jako 360°, zgodnie z konwencją meteorologiczną 1-360°. Dokumentacja Open-Meteo tego nie opisuje.

### 8.15. Stacje ze wspólną komórką siatki pogodowej

**Co widać:** kilka stacji dostało pogodę z tej samej komórki modelu i ma identyczny przebieg pogody, różni się tylko ciśnienie.

**Skala:** 19 z 96 stacji w 9 komórkach (po 2-3 stacje); identyczna temperatura, wilgotność, wiatr i opad we wszystkich 8784 godzinach. Punkt siatki leży średnio 3,8 km od stacji, najdalej 8,4 km.

**Przykład:** `DsWrocAlWisn` i `DsWrocWybCon` (Wrocław), `PkRzeszPilsu`, `PkRzeszSloci` i `PkRzeszStarz` (Rzeszów).

**Gdzie w notebooku:** 5.4

**Możliwe źródło:** Cecha źródła: Open-Meteo to model (IFS 9 km, ERA5 25 km, ERA5-Land 11 km), a nie pomiar na stacji. Domyślnie (`cell_selection=land`) wybiera komórkę lądową o wysokości podobnej do punktu. Stacje położone blisko siebie trafiają do tej samej komórki, a różni je tylko ciśnienie, które zależy od wysokości punktu.

### 8.16. Komunikat błędu zapisany jako plik danych (RAW)

**Co widać:** przy pobieraniu pogody serwer zwrócił z kodem HTTP 200 komunikat błędu zamiast danych, a skrypt zapisał go jako poprawny plik razem z sumą SHA-256. Wykryte przy pierwszym uruchomieniu notebooka 02_bronze_openmeteo; od tego czasu `scripts/download_openmeteo.py` sprawdza, czy plik to JSON z kompletną sekcją `hourly`, i pobiera go ponownie.

**Skala:** 2 z 96 plików JSON w pierwszym pobraniu.

**Przykład:** `LbZamoHrubie.json` i `LbLukBulNiepMOB.json` zawierały tekst `Unexpected error while streaming data: timeoutReached` (53 bajty).

**Gdzie w notebooku:** poza tym notebookiem: walidacja w `scripts/download_openmeteo.py`

**Możliwe źródło:** Błąd transmisji: serwer Open-Meteo wysłał już kod HTTP 200 i zaczął przesyłać dane, gdy nastąpił timeout, więc treść odpowiedzi zawierała komunikat błędu. Sam kod HTTP nie wystarcza do sprawdzenia, czy plik jest poprawny.

### Źródła informacji do sekcji 8

- GIOŚ, API - czas danych archiwalnych UTC+01:00: https://powietrze.gios.gov.pl/pjp/content/api
- GIOŚ, Bank danych pomiarowych - dane w czasie CET, dane podlegają weryfikacji: https://powietrze.gios.gov.pl/pjp/archives
- GIOŚ, komunikaty o jakości powietrza z 09-12.01.2024: https://powietrze.gios.gov.pl/pjp/content/show/1004843 (i kolejne)
- GIOŚ, powiadomienie z 27.02.2024 o ryzyku przekroczenia poziomu alarmowego SO2: https://powietrze.gios.gov.pl/pjp/rwms/content/show/11656
- 24Kurier, epizod SO2 w Szczecinie 27.02.2024: https://24kurier.pl/aktualnosci/wiadomosci/alarmujace-dane-inspekcji-ochrony-srodowiska-trujacy-dwutlenek-siarki-ponad-norme/
- Open-Meteo, Historical Weather API - modele, `cell_selection`, definicja opadu: https://open-meteo.com/en/docs/historical-weather-api
- Radio Kraków, stacje mobilne GIOŚ w uzdrowiskach: https://www.radiokrakow.pl/aktualnosci/gios-od-2019-r-monitoring-powietrza-w-polskich-uzdrowiskach

## 9. TODO - przekazanie

### Dla osoby od źródeł anomalii

- [x] Uzupełnić pole **Możliwe źródło** w każdej anomalii 8.1-8.16. Przy każdej warto oddzielić, czy to błąd techniczny (przyrząd, transmisja, zapis), czy rzeczywiste zjawisko w powietrzu, czy cecha formatu danych.
- [x] **8.9 (czas) - najważniejsze dla Milestone 2.** Znaleźć w dokumentacji GIOŚ (bank danych pomiarowych, opis archiwum), w jakiej strefie są znaczniki czasu i czy oznaczają koniec okresu. Od tego zależy, jak złączymy PM2.5 z pogodą. Wynik: strefa UTC+1 potwierdzona w dokumentacji API, koniec okresu wynika tylko z danych.
- [x] 8.8 (wartości skrajne) - ocenić, czy 8-10.01.2024 i 27.02.2024 w Szczecinie to udokumentowane epizody zanieczyszczenia (komunikaty WIOŚ/GIOŚ, wiadomości), czy błędy. Wynik: oba epizody potwierdzone komunikatami GIOŚ.
- [x] 8.3, 8.4, 8.5 (wartości ujemne, zera, stałe ciągi) - sprawdzić, jak GIOŚ i metody pomiaru traktują wartości poniżej progu wykrywalności i jakie są rozdzielczości zapisu dla gazów. Wynik: dokumentacja GIOŚ tego nie opisuje, przyczyny w 8.3-8.5 to hipotezy.
- [x] 8.6, 8.13 (piki, pomiary poza okresem) - większość dotyczy stacji mobilnych (`MOB`); sprawdzić, czym różnią się kampanie stacji mobilnych od stacji stałych.
- [x] 8.14, 8.15 (pogoda) - sprawdzić w dokumentacji Open-Meteo, jak dobierana jest komórka siatki i skąd stała wartość kierunku przy ciszy.

### Dla osoby od prezentacji

- [ ] **Twarde statystyki:** tabela zbiorcza z sekcji 7 nadaje się na slajd wprost (20 anomalii, liczba, podstawa, procent). Liczby skali zbioru: 8 952 726 pomiarów, 286 stacji, 1759 stanowisk, 60 wskaźników; pogoda 843 264 rekordy dla 96 stacji PM2.5.
- [ ] **Schemat danych:** opis tabel i kolumn z sekcji 2, relacje z diagramu w 2.5, wynik `DESCRIBE` z sekcji 2.
- [ ] **Źródło i licencja:** GIOŚ - bank danych pomiarowych (https://powietrze.gios.gov.pl/pjp/archives), warunki ponownego wykorzystania do sprawdzenia; Open-Meteo Historical Weather API - licencja CC BY 4.0 (wymaga podania źródła).
- [ ] **Anomalie:** sekcja 8 (po uzupełnieniu źródeł). Na slajd wybrać kilka najmocniejszych: czas (8.9), kody niezgodne ze słownikiem (8.10), zawieszony czujnik (8.5), wspólne komórki pogody (8.15), komunikat błędu jako dane (8.16).
- [ ] **Ograniczenia (o czym dane nie mówią):** pogoda pochodzi z modelu, a nie z pomiaru na stacji; 19 stacji ma identyczną pogodę; GIOŚ nie podaje wysokości stacji; tylko jeden rok (2024); pominięty plik depozycji (inny układ i rodzaj pomiaru, notebook 02); konwencja czasu niepotwierdzona w dokumentacji; PM2.5 godzinowo tylko na 96 stacjach.
- [ ] **Zagrożenia i etyka:** dane publiczne, bez danych osobowych; ryzyko błędnego dopasowania czasu (przesunięte cechy pogodowe); ryzyko, że czyszczenie usunie prawdziwe epizody smogowe razem z błędami.
- [ ] **Problemy i zaskoczenia przy zadaniu:** format szeroki plików GIOŚ (stanowiska jako kolumny), przesunięcie czasu GIOŚ wobec UTC, komunikat błędu API zapisany jako poprawny plik, stanowisko zdublowane w dwóch plikach, kody stacji różne w pomiarach i słowniku.
- [ ] **Dokumentacja GenAI:** prowadzący wymaga udokumentowania każdego użycia GenAI razem z promptami - zebrać prompty użyte przy profilowaniu i opisać je w prezentacji.